# Section 1: Setup

## Configuration & Constants

In [1]:
# ============================================================================
# SECTION 1: CONFIGURATION & CONSTANTS
# ============================================================================

RANDOM_SEED = 231

NUM_CLASSES = 4
PARTICLE_NAMES = ['Pion', 'Kaon', 'Proton', 'Electron']
PDG_TO_SPECIES = {
    211: 0,
    321: 1,
    2212: 2,
    11: 3,
}

MOMENTUM_RANGES = [
    {'key': 'full',        'name': 'Full Spectrum (0.1-∞ GeV/c)',          'min': 0.1, 'max': float('inf'), 'apply_dpg_cuts': False},
    {'key': '0.1-1',       'name': '0-1 GeV/c',                            'min': 0.1, 'max': 1.0,          'apply_dpg_cuts': False},
    {'key': '0.7-1.5',     'name': '0.7-1.5 GeV/c',                        'min': 0.7, 'max': 1.5,          'apply_dpg_cuts': False},
    {'key': '1-3',         'name': '1-3 GeV/c',                            'min': 1.0, 'max': 3.0,          'apply_dpg_cuts': False},
    {'key': 'full_DPG',    'name': 'Full Spectrum (0.1-∞ GeV/c, DPG cuts)', 'min': 0.1, 'max': float('inf'), 'apply_dpg_cuts': True},
    {'key': '0.1-1_DPG',   'name': '0-1 GeV/c (DPG cuts)',                  'min': 0.1, 'max': 1.0,          'apply_dpg_cuts': True},
    {'key': '0.7-1.5_DPG', 'name': '0.7-1.5 GeV/c (DPG cuts)',              'min': 0.7, 'max': 1.5,          'apply_dpg_cuts': True},
    {'key': '1-3_DPG',     'name': '1-3 GeV/c (DPG cuts)',                  'min': 1.0, 'max': 3.0,          'apply_dpg_cuts': True},
]

TRACK_SELECTIONS = {
    'event':      {'vz_max': 10.0},
    'kinematics': {'eta_min': -0.8, 'eta_max': 0.8},
    'dca':        {'dca_xy_max': 0.105, 'dca_z_max': 0.12},
    'tpc':        {'tpc_clusters_min': 70},
    'its':        {'its_clusters_min': 3},
}

TRACK_QUALITY_FILTER = {
    'tpc_clusters_min':    70,
    'momentum_min':        0.01,
    'remove_negative_chi2': True,
}

CSV_PATH = '/kaggle/input/datasets/robertforynski/new-ao2d-lhc25f60544122/pid_features_*.csv'

# ── 25 features in fixed order ────────────────────────────────────────────────
# Columns 0-20: base physics features
# Columns 21-24: per-Bayesian-feature missing indicators
# This list MUST NOT be mutated at runtime.
TRAINING_FEATURES = [
    # kinematics (3)
    'pt', 'eta', 'phi',
    # TPC (5)
    'tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el',
    # TOF (5)
    'tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el',
    # Bayesian probs (4) — sentinel -0.25 when missing
    'bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el',
    # DCA + detector flags (4)
    'dca_xy', 'dca_z',
    'has_tpc', 'has_tof',
    # Bayesian missing indicators (4) — 1=missing, 0=real
    'bayes_prob_pi_missing', 'bayes_prob_ka_missing',
    'bayes_prob_pr_missing', 'bayes_prob_el_missing',
]
assert len(TRAINING_FEATURES) == 25, f"Expected 25 features, got {len(TRAINING_FEATURES)}"

DETECTOR_GROUPS = {
    'tpc':        ['tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el'],
    'tof':        ['tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el'],
    'bayes':      ['bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el'],
    'kinematics': ['pt', 'eta', 'phi', 'dca_xy', 'dca_z'],
}
# 4 groups → group_mask shape (N, 4): tpc / tof / bayes / kinematics

# Columns that are binary or use a sentinel — skip StandardScaler for these
SKIP_SCALING = (
    {'has_tpc', 'has_tof'} |
    {f"{f}_missing" for f in DETECTOR_GROUPS['bayes']}
)

MODEL_TYPES = [
    'JAX_SimpleNN',
    'JAX_DNN',
    'JAX_FSE_Attention',
    'LightGBM',
    'XGBoost',
]

HYPERPARAMETERS = {
    'JAX_SimpleNN': {
        'hidden_dims':   [512, 256, 128, 64],
        'dropout_rate':  0.5,
        'learning_rate': 1e-4,
        'batch_size':    256,
        'num_epochs':    100,
        'patience':      30,
    },
    'JAX_DNN': {
        'hidden_dims':   [1024, 512, 256, 128, 64],
        'dropout_rate':  0.5,
        'learning_rate': 5e-5,
        'batch_size':    256,
        'num_epochs':    100,
        'patience':      30,
    },
    'JAX_FSE_Attention': {
        'hidden_dim':    128,
        'num_heads':     8,
        'dropout_rate':  0.4,
        'learning_rate': 1e-4,
        'batch_size':    256,
        'num_epochs':    100,
        'patience':      30,
    },
    'LightGBM': {
        'n_estimators':     500,
        'learning_rate':    0.05,
        'num_leaves':       64,
        'max_depth':        -1,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'random_state':     RANDOM_SEED,
        'n_jobs':           -1,
    },
    'XGBoost': {
        'n_estimators':     500,
        'learning_rate':    0.05,
        'max_depth':        6,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'tree_method':      'hist',
        'random_state':     RANDOM_SEED,
        'n_jobs':           -1,
    },
}

FORCE_TRAINING = {
    'JAX_SimpleNN':      {'full': True, '0.1-1': True, '0.7-1.5': True, '1-3': True, 'full_DPG': True, '0.1-1_DPG': True, '0.7-1.5_DPG': True, '1-3_DPG': True},
    'JAX_DNN':           {'full': False, '0.1-1': False, '0.7-1.5': False, '1-3': False, 'full_DPG': False, '0.1-1_DPG': False, '0.7-1.5_DPG': False, '1-3_DPG': False},
    'JAX_FSE_Attention': {'full': False, '0.1-1': False, '0.7-1.5': False, '1-3': False, 'full_DPG': False, '0.1-1_DPG': False, '0.7-1.5_DPG': False, '1-3_DPG': False},
    'LightGBM':          {'full': False, '0.1-1': False, '0.7-1.5': False, '1-3': False, 'full_DPG': False, '0.1-1_DPG': False, '0.7-1.5_DPG': False, '1-3_DPG': False},
    'XGBoost':           {'full': False, '0.1-1': False, '0.7-1.5': False, '1-3': False, 'full_DPG': False, '0.1-1_DPG': False, '0.7-1.5_DPG': False, '1-3_DPG': False},
}

model_colors = {
    'JAX_SimpleNN':      '#3B82F6',
    'JAX_DNN':           '#F59E0B',
    'JAX_FSE_Attention': '#22C55E',
    'Bayesian_PID':      '#F97316',
    'LightGBM':          '#14B8A6',
    'XGBoost':           '#EC4899',
}

print("✓ Configuration loaded")
print(f"  Momentum ranges:   {len(MOMENTUM_RANGES)}")
print(f"  Model types:       {len(MODEL_TYPES)}")
print(f"  Particle classes:  {NUM_CLASSES}")
print(f"  Training features: {len(TRAINING_FEATURES)}  (25 fixed)")
print(f"  Detector groups:   {len(DETECTOR_GROUPS)}  (→ 4-column group mask)")
print(f"  Skip scaling:      {sorted(SKIP_SCALING)}")


✓ Configuration loaded
  Momentum ranges:   8
  Model types:       5
  Particle classes:  4
  Training features: 25  (25 fixed)
  Detector groups:   4  (→ 4-column group mask)
  Skip scaling:      ['bayes_prob_el_missing', 'bayes_prob_ka_missing', 'bayes_prob_pi_missing', 'bayes_prob_pr_missing', 'has_tof', 'has_tpc']


## Imports

In [2]:
# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import time
import warnings
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

import glob
from pathlib import Path

import jax
import jax.numpy as jnp
from jax import random, grad, jit, vmap
import optax
from flax import linen as nn
from flax.training import train_state

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score, classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# LightGBM
import lightgbm as lgb

# XGBoost
import xgboost as xgb

print(f"✓ JAX version: {jax.__version__}")
print(f"✓ Available devices: {jax.devices()}")
print(f"✓ All libraries imported\n")

print(f"{'='*80}")
print("✓ SECTION 1 COMPLETE")
print(f"{'='*80}\n")


✓ JAX version: 0.7.2
✓ Available devices: [CudaDevice(id=0), CudaDevice(id=1)]
✓ All libraries imported

✓ SECTION 1 COMPLETE



# Section 2: Data Loading & Preprocessing Utilities

In [3]:
# ============================================================================
# SECTION 2: DATA LOADING & PREPROCESSING UTILITIES
# ============================================================================

def load_data(csv_pattern, verbose=True):
    csv_files = sorted(glob.glob(csv_pattern))
    if not csv_files:
        raise FileNotFoundError(f"No files matching: {csv_pattern}")

    if verbose:
        print(f"Searching for CSV files matching:\n  {csv_pattern}")
        print(f"\nFound {len(csv_files)} CSV files:\n")

    dfs = []
    for file_idx, csv_file in enumerate(csv_files, 1):
        file_size_mb = Path(csv_file).stat().st_size / (1024**2)
        filename = Path(csv_file).name
        if verbose:
            print(f"  {file_idx}. {filename} ({file_size_mb:.1f} MB)... ", end='', flush=True)
        try:
            df = pd.read_csv(
                csv_file,
                dtype='float32',
                na_values=['-', 'nan', 'null', 'NaN', 'NULL', ''],
                keep_default_na=True,
                on_bad_lines='skip',
                low_memory=False,
            )
            if verbose:
                print(f"✓ Loaded {len(df):,} rows")
            dfs.append(df)
        except Exception as e:
            if verbose:
                print(f"✗ ERROR: {str(e)[:60]}")
            continue

    if not dfs:
        raise ValueError("No data loaded from any file!")

    if verbose:
        print(f"\nCombining {len(dfs)} files...")

    df_combined = pd.concat(dfs, ignore_index=True, sort=False)

    if verbose:
        print(f"✓ Combined shape: {df_combined.shape}")
        print(f"  Total rows: {len(df_combined):,}")
        print(f"  Total columns: {df_combined.shape[1]}\n")

    print(f"Total tracks loaded from CSVs (all PDGs): {len(df_combined):,}")
    return df_combined


def pdg_to_species(pdg):
    if pd.isna(pdg):
        return -1
    try:
        return PDG_TO_SPECIES.get(abs(int(pdg)), -1)
    except (ValueError, TypeError):
        return -1


def preprocess_momentum_range(df, momentum_range):
    """
    Preprocess data for one momentum range.

    Feature contract (25 features, fixed order — see TRAINING_FEATURES):
      [0-2]   pt, eta, phi
      [3-7]   tpc_signal, tpc_nsigma_pi/ka/pr/el
      [8-12]  tof_beta, tof_nsigma_pi/ka/pr/el
      [13-16] bayes_prob_pi/ka/pr/el  (sentinel -0.25 when missing)
      [17-20] dca_xy, dca_z, has_tpc, has_tof
      [21-24] bayes_prob_pi/ka/pr/el_missing  (1=missing, 0=real)

    Group mask (N, 4) — fixed column order matching DETECTOR_GROUPS:
      col 0: has_tpc           (TPC group active)
      col 1: has_tof            (TOF group active)
      col 2: bayes_present      (1 if at least one bayes feature is real)
      col 3: ones               (kinematics always active)
    """
    mr_key      = momentum_range['key']
    use_dpg     = momentum_range.get('apply_dpg_cuts', False)
    bayes_feats = DETECTOR_GROUPS['bayes']

    print(f"\n{'─'*80}")
    print(f"Preprocessing: {momentum_range['name']}  [key={mr_key}]")
    print(f"{'─'*80}")
    print(f"  DPG cuts : {'YES' if use_dpg else 'NO'}")
    print(f"  Features : {len(TRAINING_FEATURES)} (fixed)")
    print(f"  Groups   : {len(DETECTOR_GROUPS)} (tpc/tof/bayes/kinematics)")

    # ── STEP 1: momentum filter ───────────────────────────────────────────
    df_f = df[(df['p'] >= momentum_range['min']) &
              (df['p'] <  momentum_range['max'])].copy()
    print(f"\n  After momentum filter: {len(df_f):,}")

    # ── STEP 2: DPG cuts (optional) ───────────────────────────────────────
    if use_dpg:
        print(f"\n  Applying DPG-recommended track selections:")
        n_before = len(df_f)

        if 'vz' in df_f.columns:
            df_f = df_f[df_f['vz'].abs() < TRACK_SELECTIONS['event']['vz_max']]

        eta_min = TRACK_SELECTIONS['kinematics']['eta_min']
        eta_max = TRACK_SELECTIONS['kinematics']['eta_max']
        if 'eta' in df_f.columns:
            df_f = df_f[df_f['eta'].between(eta_min, eta_max, inclusive='both')]

        if 'dca_xy' in df_f.columns:
            df_f = df_f[df_f['dca_xy'].abs() < TRACK_SELECTIONS['dca']['dca_xy_max']]
        if 'dca_z' in df_f.columns:
            df_f = df_f[df_f['dca_z'].abs()  < TRACK_SELECTIONS['dca']['dca_z_max']]

        tpc_col = next((c for c in ['tpc_nclusters', 'tpc_ncls'] if c in df_f.columns), None)
        if tpc_col:
            df_f = df_f[df_f[tpc_col] >= TRACK_SELECTIONS['tpc']['tpc_clusters_min']]

        its_col = next((c for c in ['its_nclusters', 'its_ncls'] if c in df_f.columns), None)
        if its_col:
            df_f = df_f[df_f[its_col] >= TRACK_SELECTIONS['its']['its_clusters_min']]

        if TRACK_QUALITY_FILTER.get('remove_negative_chi2'):
            for chi2_col in ['tpc_chi2', 'track_chi2', 'chi2_per_ndf', 'chi2']:
                if chi2_col in df_f.columns:
                    df_f = df_f[df_f[chi2_col] >= 0]
                    break

        df_f = df_f[df_f['p'] >= TRACK_QUALITY_FILTER['momentum_min']]

        print(f"  After DPG selections: {len(df_f):,}  "
              f"(removed {n_before - len(df_f):,}, "
              f"{100*(n_before-len(df_f))/max(n_before,1):.2f}%)")

    # ── STEP 3: imputation ─────────────────────────────────────────────────
    for feat in DETECTOR_GROUPS['tof']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(0.0 if feat == 'tof_beta' else 999.0)

    for feat in DETECTOR_GROUPS['tpc']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(0.0 if feat == 'tpc_signal' else 999.0)

    for feat in DETECTOR_GROUPS['kinematics']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(df_f[feat].median())

    for col in ['has_tpc', 'has_tof']:
        if col in df_f.columns:
            df_f[col] = df_f[col].fillna(0.0)
        else:
            df_f[col] = 0.0

    # ── STEP 4: Bayesian — record real/missing BEFORE sentinel fill ────────
    print(f"\n  Bayesian PID status (raw data):")
    bayes_real_row = (df_f[bayes_feats] > 0).all(axis=1)   # True = real
    n_real  = bayes_real_row.sum()
    n_total = len(df_f)
    print(f"    REAL  (>0):  {n_real:,} ({100*n_real/n_total:.2f}%)")
    print(f"    MISSING:     {n_total-n_real:,} ({100*(n_total-n_real)/n_total:.2f}%)")

    # Store originals for Bayesian comparison analysis
    bayes_complete_before = bayes_real_row.values          # (N,) bool
    bayes_probs_before    = df_f[bayes_feats].values.astype('float32')  # (N,4)

    # Create per-feature missing indicator columns + apply sentinel
    for feat in bayes_feats:
        miss_col = f"{feat}_missing"
        missing  = (df_f[feat] == 0) | df_f[feat].isna()
        df_f[miss_col]      = missing.astype('float32')
        df_f.loc[missing, feat] = -0.25
        # NOTE: do NOT append to TRAINING_FEATURES — already defined in Section 1

    # Drop any remaining NaN in the feature columns
    nan_count = df_f[TRAINING_FEATURES].isnull().sum().sum()
    if nan_count > 0:
        print(f"\n  ⚠ Dropping {nan_count} residual NaN values...")
        df_f = df_f.dropna(subset=TRAINING_FEATURES)

    # ── STEP 5: PDG → species label ───────────────────────────────────────
    y_all   = np.array([pdg_to_species(p) for p in df_f['mc_pdg'].values])
    valid   = y_all >= 0
    n_kept  = valid.sum()
    print(f"\n  PDG filter: kept {n_kept:,} / {len(y_all):,}  "
          f"(dropped {len(y_all)-n_kept:,})")

    bayes_complete_before = bayes_complete_before[valid]
    bayes_probs_before    = bayes_probs_before[valid]

    # ── STEP 6: build feature matrix (strict column order) ─────────────────
    missing_cols = [f for f in TRAINING_FEATURES if f not in df_f.columns]
    if missing_cols:
        raise ValueError(f"Missing columns after preprocessing: {missing_cols}")

    X_all = df_f[TRAINING_FEATURES].values[valid].astype('float32')
    y_all = y_all[valid]
    assert X_all.shape[1] == 25, f"Expected 25 features, got {X_all.shape[1]}"

    # ── STEP 7: group masks (4 columns, fixed order) ───────────────────────
    tpc_idx = TRAINING_FEATURES.index('has_tpc')
    tof_idx = TRAINING_FEATURES.index('has_tof')

    has_tpc_arr = (X_all[:, tpc_idx] >= 0.5).astype('float32')
    has_tof_arr = (X_all[:, tof_idx] >= 0.5).astype('float32')

    # Bayes group active = NOT all four missing-indicator columns equal to 1
    miss_indices = [TRAINING_FEATURES.index(f"{f}_missing") for f in bayes_feats]
    all_missing  = np.ones(len(X_all), dtype='float32')
    for mi in miss_indices:
        all_missing *= X_all[:, mi]       # 1 only when ALL are missing
    bayes_present = (1.0 - all_missing)   # 1 when at least one is real

    group_masks = np.stack([
        has_tpc_arr,                                        # col 0: TPC
        has_tof_arr,                                        # col 1: TOF
        bayes_present,                                      # col 2: Bayes
        np.ones(len(X_all), dtype='float32'),               # col 3: kinematics
    ], axis=1)
    assert group_masks.shape[1] == 4

    # ── STEP 8: detector modes (for analysis only) ─────────────────────────
    detector_mode = np.zeros(len(X_all), dtype='int32')
    detector_mode[(has_tpc_arr == 1) & (has_tof_arr == 0)] = 1  # TPC_ONLY
    detector_mode[(has_tpc_arr == 0) & (has_tof_arr == 1)] = 2  # TOF_ONLY
    detector_mode[(has_tpc_arr == 1) & (has_tof_arr == 1)] = 3  # TPC_TOF

    # ── STEP 9: stratified train/test split ───────────────────────────────
    (X_train, X_test,
     y_train, y_test,
     masks_train, masks_test,
     bayes_mask_train, bayes_mask_test,
     bayes_probs_train, bayes_probs_test,
     modes_train, modes_test) = train_test_split(
        X_all, y_all,
        group_masks,
        bayes_complete_before.astype('float32'),
        bayes_probs_before,
        detector_mode,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=y_all,
    )
    print(f"\n  Train: {len(X_train):,}  |  Test: {len(X_test):,}")

    # ── STEP 10: per-feature scaling (sentinel & binary columns protected) ─
    feature_scalers  = {}
    X_train_scaled   = X_train.copy()
    X_test_scaled    = X_test.copy()

    for i, feat in enumerate(TRAINING_FEATURES):
        if feat in SKIP_SCALING:
            continue   # binary / missing-indicator: leave as 0/1 or 0.0

        col_tr     = X_train[:, i]
        feat_scaler = StandardScaler()

        if feat in bayes_feats:
            # Scale only real values; sentinel (-0.25) stays untouched
            real_tr = col_tr != -0.25
            if real_tr.sum() > 0:
                feat_scaler.fit(col_tr[real_tr].reshape(-1, 1))
                X_train_scaled[real_tr, i] = feat_scaler.transform(
                    col_tr[real_tr].reshape(-1, 1)).reshape(-1)
                real_te = X_test[:, i] != -0.25
                if real_te.sum() > 0:
                    X_test_scaled[real_te, i] = feat_scaler.transform(
                        X_test[real_te, i].reshape(-1, 1)).reshape(-1)
        else:
            feat_scaler.fit(col_tr.reshape(-1, 1))
            X_train_scaled[:, i] = feat_scaler.transform(
                col_tr.reshape(-1, 1)).reshape(-1)
            X_test_scaled[:, i]  = feat_scaler.transform(
                X_test[:, i].reshape(-1, 1)).reshape(-1)

        feature_scalers[feat] = feat_scaler

    print(f"  ✓ Per-feature scalers fitted: {len(feature_scalers)}")
    print(f"  ✓ Skipped (binary/sentinel):  {sorted(SKIP_SCALING)}")

    # ── STEP 11: sample weights ────────────────────────────────────────────
    # Tracks with real Bayesian data get 3× weight
    bayes_weight          = 1.0   # conservative; increase for Bayesian-rich ranges
    sample_weights_train  = np.where(
        bayes_mask_train > 0, bayes_weight * 3.0, 1.0
    ).astype('float32')
    sample_weights_train /= sample_weights_train.mean()

    sample_weights_test   = np.where(
        bayes_mask_test > 0, bayes_weight * 3.0, 1.0
    ).astype('float32')
    sample_weights_test  /= sample_weights_test.mean()

    print(f"  Sample weights: real_bayes={bayes_weight*3.0:.1f}x  filled=1.0x")

    # ── STEP 12: reporting ─────────────────────────────────────────────────
    print(f"\n  Particle distribution (full range):")
    for cls, cnt in zip(*np.unique(y_all, return_counts=True)):
        print(f"    {PARTICLE_NAMES[cls]:10s}: {cnt:8,d} ({100*cnt/len(y_all):6.2f}%)")

    print(f"\n  Particle distribution (train set):")
    for cls, cnt in zip(*np.unique(y_train, return_counts=True)):
        print(f"    {PARTICLE_NAMES[cls]:10s}: {cnt:8,d} ({100*cnt/len(y_train):6.2f}%)")

    mode_names = {0: 'No detector', 1: 'TPC-only', 2: 'TOF-only', 3: 'TPC+TOF'}
    print(f"\n  Detector mode distribution (train set):")
    for mode, cnt in zip(*np.unique(modes_train, return_counts=True)):
        print(f"    {mode_names.get(mode,str(mode)):12s}: {cnt:8,d} ({100*cnt/len(modes_train):6.2f}%)")

    print(f"\n{'─'*80}")

    return {
        # scaled arrays (model input)
        'X_train_scaled':           X_train_scaled,
        'X_test_scaled':            X_test_scaled,
        # unscaled arrays (mask building, detector-mode analysis)
        'X_train':                  X_train,
        'X_test':                   X_test,
        'y_train':                  y_train,
        'y_test':                   y_test,
        # ── inference contract ─────────────────────────────────────────
        'features':                 list(TRAINING_FEATURES),   # 25-item ordered list
        'feature_scalers':          feature_scalers,           # {feat: fitted StandardScaler}
        'num_groups':               len(DETECTOR_GROUPS),      # 4
        'group_names':              list(DETECTOR_GROUPS.keys()),
        'skip_scaling':             list(SKIP_SCALING),
        # ── masks & modes ──────────────────────────────────────────────
        'masks_train':              masks_train,
        'masks_test':               masks_test,
        'group_names':              list(DETECTOR_GROUPS.keys()),
        'detector_modes_train':     modes_train,
        'detector_modes_test':      modes_test,
        # ── Bayesian comparison data ───────────────────────────────────
        'bayes_availability_train': bayes_mask_train,
        'bayes_availability_test':  bayes_mask_test,
        'bayes_pred_original_train': bayes_probs_train,
        'bayes_pred_original_test':  bayes_probs_test,
        # ── sample weights ─────────────────────────────────────────────
        'sample_weights_train':     sample_weights_train,
        'sample_weights_test':      sample_weights_test,
        'bayes_weight':             bayes_weight,
    }


# ============================================================================
# 2.2: MODEL PERSISTENCE
# ============================================================================

def get_model_path(momentum_range_key, model_type, mode='save'):
    model_subdir = "trained_models"
    working_path = f"/kaggle/working/{model_subdir}/{momentum_range_key}_{model_type}.pkl"
    input_path   = f"/kaggle/input/jax-models/jax-models/{momentum_range_key}_{model_type}.pkl"
    if mode == "save":
        return working_path
    return working_path if os.path.exists(working_path) else input_path


def load_single_model(momentum_range_key, model_type):
    path = get_model_path(momentum_range_key, model_type, mode="load")
    if os.path.exists(path):
        try:
            with open(path, 'rb') as f:
                results = pickle.load(f)
            print(f"✓ Loaded from: {path}")
            return results, path
        except Exception as e:
            print(f"Error loading {path}: {e}")
    return None, path


def save_single_model(momentum_range_key, model_type, results):
    """Save model results, always embedding the full inference contract."""
    path = get_model_path(momentum_range_key, model_type, mode="save")
    os.makedirs(os.path.dirname(path), exist_ok=True)

    # Embed inference contract (safe to overwrite if already present)
    results['features']     = list(TRAINING_FEATURES)
    results['num_groups']   = len(DETECTOR_GROUPS)
    results['group_names']  = list(DETECTOR_GROUPS.keys())
    results['skip_scaling'] = list(SKIP_SCALING)

    try:
        with open(path, 'wb') as f:
            pickle.dump(results, f)
        print(f"✓ Saved to: {path}")
        print(f"  features   : {len(results['features'])}")
        print(f"  num_groups : {results['num_groups']}")
        n_scalers = len(results.get('feature_scalers', {}))
        print(f"  scalers    : {n_scalers}")
    except Exception as e:
        print(f"Error saving to {path}: {e}")


print("✓ Data loading & preprocessing utilities defined")
print("  • 25-feature fixed contract")
print("  • 4-column group mask (tpc/tof/bayes/kinematics)")
print("  • Per-feature scalers (sentinel + binary protected)")
print("  • Sample weights computed")
print("  • Full inference contract saved with every model")
print(f"\n{'='*80}")
print("✓ SECTION 2 COMPLETE")
print(f"{'='*80}\n")


✓ Data loading & preprocessing utilities defined
  • 25-feature fixed contract
  • 4-column group mask (tpc/tof/bayes/kinematics)
  • Per-feature scalers (sentinel + binary protected)
  • Sample weights computed
  • Full inference contract saved with every model

✓ SECTION 2 COMPLETE



# Section 3: Model Definitions & Training Functions

In [4]:
# ============================================================================
# SECTION 3: MODEL DEFINITIONS & TRAINING UTILITIES
# ============================================================================

# ── 3.1: Focal loss ──────────────────────────────────────────────────────────

def focal_loss(logits, labels, alpha=0.25, gamma=2.0):
    log_probs   = jax.nn.log_softmax(logits, axis=-1)
    log_pt      = log_probs[jnp.arange(labels.shape[0]), labels]
    pt          = jnp.clip(jnp.exp(log_pt), 1e-8, 1.0)
    focal_weight = alpha * (1.0 - pt) ** gamma
    return jnp.mean(-focal_weight * log_pt)

# ── 3.2: Model architectures ─────────────────────────────────────────────────

class JAX_SimpleNN(nn.Module):
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, training=False):
        for dim in self.hidden_dims:
            x = nn.Dense(dim)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class JAX_DNN(nn.Module):
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, training=False):
        for dim in self.hidden_dims:
            x = nn.Dense(dim)(x)
            x = nn.BatchNorm(use_running_average=not training)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class JAX_FSE_Attention(nn.Module):
    hidden_dim:   int   = 64
    num_heads:    int   = 4
    num_classes:  int   = 4
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, group_mask, training=False):
        batch_size = x.shape[0]
        num_groups = group_mask.shape[1]   # 4

        feat_proj = nn.Dense(self.hidden_dim * num_groups)(x)
        feat_proj = feat_proj.reshape(batch_size, num_groups, self.hidden_dim)
        feat_proj = feat_proj * group_mask[:, :, None]

        attn_mask = group_mask[:, None, None, :]
        feat_attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(
            feat_proj, feat_proj, mask=attn_mask
        )
        feat_attn = nn.LayerNorm()(feat_attn)

        gates     = nn.sigmoid(nn.Dense(self.hidden_dim)(feat_attn))
        feat_gated = feat_attn * gates

        denom  = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), 1.0)
        pooled = jnp.sum(feat_gated * group_mask[:, :, None], axis=1) / denom

        x = nn.Dense(128)(pooled);  x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(64)(x);        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)

# ── 3.3: Train step functions ─────────────────────────────────────────────────

@jit
def train_step_simple(state, batch_x, batch_y, rng):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x,
                                training=True, rngs={'dropout': rng})
        return focal_loss(logits, batch_y)
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    return state.apply_gradients(grads=grads), loss


@jit
def train_step_batchnorm(state, batch_x, batch_y, rng):
    def loss_fn(params):
        variables = {'params': params, 'batch_stats': state.batch_stats}
        logits, new_state = state.apply_fn(variables, batch_x, training=True,
                                           rngs={'dropout': rng},
                                           mutable=['batch_stats'])
        return focal_loss(logits, batch_y), new_state
    (loss, new_state), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    return state.replace(batch_stats=new_state['batch_stats']), loss


@jit
def train_step_fse(state, batch_x, batch_mask, batch_y, rng):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x, batch_mask,
                                training=True, rngs={'dropout': rng})
        return focal_loss(logits, batch_y)
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    return state.apply_gradients(grads=grads), loss

# ── 3.4: Evaluation functions ─────────────────────────────────────────────────

@jit
def eval_step_simple(state, batch_x, batch_y):
    logits = state.apply_fn({'params': state.params}, batch_x, training=False)
    return jnp.mean(jnp.argmax(logits, -1) == batch_y), logits

@jit
def eval_step_batchnorm(state, batch_x, batch_y):
    logits = state.apply_fn({'params': state.params, 'batch_stats': state.batch_stats},
                            batch_x, training=False)
    return jnp.mean(jnp.argmax(logits, -1) == batch_y), logits

@jit
def eval_step_fse(state, batch_x, batch_mask, batch_y):
    logits = state.apply_fn({'params': state.params}, batch_x, batch_mask, training=False)
    return jnp.mean(jnp.argmax(logits, -1) == batch_y), logits


def _batch_eval(eval_fn, *arrays, batch_size=4096):
    """Generic batched evaluation — returns (mean_acc, all_logits)."""
    all_logits, all_accs = [], []
    n = len(arrays[0])
    for start in range(0, n, batch_size):
        end    = min(start + batch_size, n)
        slices = tuple(a[start:end] for a in arrays)
        acc, logits = eval_fn(*slices)
        all_logits.append(logits)
        all_accs.append(float(acc))
    return float(np.mean(all_accs)), jnp.concatenate(all_logits, axis=0)

def batch_evaluate_simple(state, X, y, batch_size=4096):
    return _batch_eval(lambda bx, by: eval_step_simple(state, bx, by),
                       X, y, batch_size=batch_size)

def batch_evaluate_batchnorm(state, X, y, batch_size=4096):
    return _batch_eval(lambda bx, by: eval_step_batchnorm(state, bx, by),
                       X, y, batch_size=batch_size)

def batch_evaluate_fse(state, X, masks, y, batch_size=4096):
    return _batch_eval(lambda bx, bm, by: eval_step_fse(state, bx, bm, by),
                       X, masks, y, batch_size=batch_size)

# ── 3.5: Extended TrainState for BatchNorm ────────────────────────────────────

class TrainStateWithBatchStats(train_state.TrainState):
    batch_stats: any = None

# ── 3.6: Unified training orchestrator ───────────────────────────────────────

def train_model(model_type, momentum_range, preprocessing_data, force_training=False):
    mr_key = momentum_range['key']
    hp     = HYPERPARAMETERS[model_type]

    print(f"\n{'*'*80}")
    print(f"{model_type} — {momentum_range['name']}")
    print(f"{'*'*80}\n")

    # ── try to load cached model ──────────────────────────────────────────
    if not force_training:
        loaded, _ = load_single_model(mr_key, model_type)
        if loaded is not None:
            print("✓ Loaded existing model (skipped training)")
            return loaded

    print("Training from scratch...")
    for k, v in hp.items():
        print(f"    {k:20s}: {v}")

    X_train = preprocessing_data['X_train_scaled']
    X_test  = preprocessing_data['X_test_scaled']
    y_train = preprocessing_data['y_train']
    y_test  = preprocessing_data['y_test']
    X_test_raw = preprocessing_data.get('X_test', X_test)   # unscaled

    X_tr_jax = jnp.array(X_train, dtype=jnp.float32)
    X_te_jax = jnp.array(X_test,  dtype=jnp.float32)
    y_tr_jax = jnp.array(y_train, dtype=jnp.int32)
    y_te_jax = jnp.array(y_test,  dtype=jnp.int32)

    print(f"✓ Data: train {X_tr_jax.shape}, test {X_te_jax.shape}")

    key = random.PRNGKey(RANDOM_SEED + hash(model_type) % 10000)

    # ── initialise model ──────────────────────────────────────────────────
    if model_type == 'JAX_SimpleNN':
        model       = JAX_SimpleNN(hidden_dims=hp['hidden_dims'],
                                   num_classes=NUM_CLASSES,
                                   dropout_rate=hp['dropout_rate'])
        dummy       = jnp.ones((1, X_tr_jax.shape[1]))
        mp          = model.init(key, dummy, training=False)
        tx          = optax.chain(optax.clip_by_global_norm(1.0),
                                  optax.adam(hp['learning_rate']))
        state       = train_state.TrainState.create(
                          apply_fn=model.apply, params=mp['params'], tx=tx)
        train_fn    = train_step_simple

    elif model_type == 'JAX_DNN':
        model       = JAX_DNN(hidden_dims=hp['hidden_dims'],
                              num_classes=NUM_CLASSES,
                              dropout_rate=hp['dropout_rate'])
        dummy       = jnp.ones((1, X_tr_jax.shape[1]))
        variables   = model.init(key, dummy, training=True)
        tx          = optax.chain(optax.clip_by_global_norm(1.0),
                                  optax.adam(hp['learning_rate']))
        state       = TrainStateWithBatchStats.create(
                          apply_fn=model.apply,
                          params=variables['params'],
                          tx=tx,
                          batch_stats=variables.get('batch_stats', {}))
        train_fn    = train_step_batchnorm

    elif model_type == 'JAX_FSE_Attention':
        model       = JAX_FSE_Attention(hidden_dim=hp['hidden_dim'],
                                        num_heads=hp['num_heads'],
                                        num_classes=NUM_CLASSES,
                                        dropout_rate=hp['dropout_rate'])
        masks_tr_jax = jnp.array(preprocessing_data['masks_train'], dtype=jnp.float32)
        masks_te_jax = jnp.array(preprocessing_data['masks_test'],  dtype=jnp.float32)
        dummy_x     = jnp.ones((1, X_tr_jax.shape[1]))
        dummy_m     = jnp.ones((1, masks_tr_jax.shape[1]))
        mp          = model.init(key, dummy_x, dummy_m, training=False)
        tx          = optax.chain(optax.clip_by_global_norm(1.0),
                                  optax.adam(hp['learning_rate']))
        state       = train_state.TrainState.create(
                          apply_fn=model.apply, params=mp['params'], tx=tx)
        train_fn    = train_step_fse

    print("✓ Model initialised")

    # ── training loop ─────────────────────────────────────────────────────
    num_batches       = len(X_tr_jax) // hp['batch_size']
    best_val_acc      = 0.0
    patience_cnt      = 0
    train_losses      = []
    train_accuracies  = []
    val_accuracies    = []
    epoch_times       = []
    best_params       = None
    best_batch_stats  = None
    main_key          = key

    training_start_time = time.time()

    print(f"\nTraining (max {hp['num_epochs']} epochs, patience={hp['patience']})...\n")

    for epoch in range(hp['num_epochs']):

        epoch_start_time = time.time()

        main_key, shuffle_key, dropout_key = random.split(main_key, 3)
        perm              = random.permutation(shuffle_key, len(X_tr_jax))
        X_shuf            = X_tr_jax[perm]
        y_shuf            = y_tr_jax[perm]

        if model_type == 'JAX_FSE_Attention':
            m_shuf = masks_tr_jax[perm]

        epoch_losses = []

        for bi in range(num_batches):
            dropout_key, subkey = random.split(dropout_key)
            s   = bi * hp['batch_size']
            e   = s  + hp['batch_size']
            bx  = X_shuf[s:e]
            by  = y_shuf[s:e]

            if model_type == 'JAX_FSE_Attention':
                state, loss = train_fn(state, bx, m_shuf[s:e], by, subkey)
            else:
                state, loss = train_fn(state, bx, by, subkey)

            epoch_losses.append(float(loss))

        avg_loss = float(np.mean(epoch_losses))
        train_losses.append(avg_loss)

        # ── compute train accuracy each epoch ─────────────────────────────
        if model_type == 'JAX_FSE_Attention':
            train_acc_epoch, _ = batch_evaluate_fse(state, X_tr_jax, masks_tr_jax, y_tr_jax)
            val_acc, _         = batch_evaluate_fse(state, X_te_jax, masks_te_jax, y_te_jax)
        elif model_type == 'JAX_DNN':
            train_acc_epoch, _ = batch_evaluate_batchnorm(state, X_tr_jax, y_tr_jax)
            val_acc, _         = batch_evaluate_batchnorm(state, X_te_jax, y_te_jax)
        else:
            train_acc_epoch, _ = batch_evaluate_simple(state, X_tr_jax, y_tr_jax)
            val_acc, _         = batch_evaluate_simple(state, X_te_jax, y_te_jax)

        train_accuracies.append(float(train_acc_epoch))
        val_accuracies.append(float(val_acc))

        epoch_elapsed = time.time() - epoch_start_time
        epoch_times.append(epoch_elapsed)

        if (epoch == 0) or ((epoch + 1) % 10 == 0) or (epoch + 1 == hp['num_epochs']):
            print(f"Epoch {epoch+1:3d}/{hp['num_epochs']} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Train Acc: {train_acc_epoch:.4f} | "
                  f"Val Acc: {val_acc:.4f} | "
                  f"Time: {epoch_elapsed:.2f}s")

        if val_acc > best_val_acc:
            best_val_acc  = val_acc
            patience_cnt  = 0
            best_params   = state.params
            if model_type == 'JAX_DNN':
                best_batch_stats = state.batch_stats
        else:
            patience_cnt += 1
            if patience_cnt >= hp['patience']:
                print(f"✓ Early stopping at epoch {epoch+1}")
                break

    total_training_time = time.time() - training_start_time
    trained_epochs      = len(train_losses)

    # restore best
    state = state.replace(params=best_params)
    if model_type == 'JAX_DNN':
        state = state.replace(batch_stats=best_batch_stats)

    # ── final evaluation ──────────────────────────────────────────────────
    print(f"\nFinal evaluation (test N={len(X_te_jax):,})...")

    if model_type == 'JAX_FSE_Attention':
        train_acc, train_logits = batch_evaluate_fse(state, X_tr_jax, masks_tr_jax, y_tr_jax)
        test_acc,  test_logits  = batch_evaluate_fse(state, X_te_jax, masks_te_jax, y_te_jax)
    elif model_type == 'JAX_DNN':
        train_acc, train_logits = batch_evaluate_batchnorm(state, X_tr_jax, y_tr_jax)
        test_acc,  test_logits  = batch_evaluate_batchnorm(state, X_te_jax, y_te_jax)
    else:
        train_acc, train_logits = batch_evaluate_simple(state, X_tr_jax, y_tr_jax)
        test_acc,  test_logits  = batch_evaluate_simple(state, X_te_jax, y_te_jax)

    train_probs  = np.array(jax.nn.softmax(train_logits, axis=-1))
    test_probs   = np.array(jax.nn.softmax(test_logits,  axis=-1))
    y_pred_test  = np.array(jnp.argmax(test_logits, axis=-1))

    print(f"  Train Acc           : {train_acc:.4f}")
    print(f"  Test Acc            : {test_acc:.4f}")
    print(f"  Best Validation Acc : {best_val_acc:.4f}")
    print(f"  Trained Epochs      : {trained_epochs}")
    print(f"  Total Train Time    : {total_training_time:.2f} sec")
    print(f"  Avg Epoch Time      : {np.mean(epoch_times):.2f} sec")
    print(f"  Fastest Epoch       : {np.min(epoch_times):.2f} sec")
    print(f"  Slowest Epoch       : {np.max(epoch_times):.2f} sec")

    # ── detector modes on test set ────────────────────────────────────────
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')
        has_tpc_bin = (np.array(X_test_raw[:, idx_tpc]) >= 0.5).astype(int)
        has_tof_bin = (np.array(X_test_raw[:, idx_tof]) >= 0.5).astype(int)
        detector_modes          = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
    except Exception as ex:
        print(f"  ⚠ Could not build detector_modes: {ex}")
        detector_modes = None

    # ── build results dict with full inference contract ───────────────────
    results = {
        'model_type':      model_type,
        'hyperparameters': hp,

        'train_losses':    train_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies':  val_accuracies,
        'epoch_times':     epoch_times,

        'trained_epochs':      trained_epochs,
        'total_train_time':    float(total_training_time),
        'avg_epoch_time':      float(np.mean(epoch_times)),
        'fastest_epoch_time':  float(np.min(epoch_times)),
        'slowest_epoch_time':  float(np.max(epoch_times)),

        'best_val_acc':    float(best_val_acc),
        'train_acc':       float(train_acc),
        'test_acc':        float(test_acc),

        'train_probs':     train_probs,
        'test_probs':      test_probs,
        'y_pred_test':     y_pred_test,
        'y_test':          np.array(y_te_jax),

        # ── inference contract ─────────────────────────────────────────
        'params':          best_params,
        'features':        list(TRAINING_FEATURES),
        'num_groups':      len(DETECTOR_GROUPS),
        'group_names':     list(DETECTOR_GROUPS.keys()),
        'feature_scalers': preprocessing_data.get('feature_scalers', {}),
        'skip_scaling':    list(SKIP_SCALING),
    }

    if model_type == 'JAX_DNN':
        results['batch_stats'] = best_batch_stats

    if detector_modes is not None:
        results['detector_modes'] = detector_modes

    save_single_model(mr_key, model_type, results)
    return results


print("✓ Focal loss, model architectures, train/eval functions defined")
print(f"\n{'='*80}")
print("✓ SECTION 3 COMPLETE")
print(f"{'='*80}\n")


✓ Focal loss, model architectures, train/eval functions defined

✓ SECTION 3 COMPLETE



# Section 4: Data Loading & Initialisation

In [5]:
# ============================================================================
# SECTION 4: DATA LOADING & INITIALISATION
# ============================================================================

# Load data once
df = load_data(CSV_PATH)

print(f"\n✓ Data loaded")
print(f"Total tracks in dataset: {len(df):,}")

# ------------------------------------------------------------------
# Helper: apply DPG cuts
# ------------------------------------------------------------------
def apply_dpg_track_cuts(df):
    df_cut = df.copy()

    # Eta cut
    if 'eta' in df_cut.columns:
        df_cut = df_cut[
            (df_cut['eta'] > TRACK_SELECTIONS['kinematics']['eta_min']) &
            (df_cut['eta'] < TRACK_SELECTIONS['kinematics']['eta_max'])
        ]

    # DCA cuts
    if 'dca_xy' in df_cut.columns and 'dca_z' in df_cut.columns:
        df_cut = df_cut[
            (np.abs(df_cut['dca_xy']) < TRACK_SELECTIONS['dca']['dca_xy_max']) &
            (np.abs(df_cut['dca_z']) < TRACK_SELECTIONS['dca']['dca_z_max'])
        ]
    else:
        print("  ⚠ DCA columns not present → skipping DCA cuts")

    return df_cut

# ------------------------------------------------------------------
# Count tracks for each momentum range
# ------------------------------------------------------------------
print("\nTrack counts by momentum range:\n")

no_dpg_ranges = [r for r in MOMENTUM_RANGES if not r['apply_dpg_cuts']]
dpg_ranges = [r for r in MOMENTUM_RANGES if r['apply_dpg_cuts']]

# No DPG cuts
print("NO DPG CUTS")
print("-"*70)
for r in no_dpg_ranges:
    df_range = df[
        (df['pt'] >= r['min']) &
        (df['pt'] < r['max'])
    ]
    print(f"{r['key']:12s} ({r['name']:30s}) : {len(df_range):,} tracks")

# DPG cuts
print("\nDPG CUTS APPLIED")
print("-"*70)
df_dpg = apply_dpg_track_cuts(df)
for r in dpg_ranges:
    df_range = df_dpg[
        (df_dpg['pt'] >= r['min']) &
        (df_dpg['pt'] < r['max'])
    ]
    print(f"{r['key']:12s} ({r['name']:30s}) : {len(df_range):,} tracks")

# Initialise master results storage
all_results_by_model_and_range = {}

print("\n✓ Results storage initialised")
print(f"\n{'='*80}")
print("✓ SECTION 4 COMPLETE: Ready for training")
print(f"{'='*80}\n")


Searching for CSV files matching:
  /kaggle/input/datasets/robertforynski/new-ao2d-lhc25f60544122/pid_features_*.csv

Found 9 CSV files:

  1. pid_features_001.csv (693.8 MB)... ✓ Loaded 3,278,355 rows
  2. pid_features_002.csv (628.0 MB)... ✓ Loaded 2,968,161 rows
  3. pid_features_003.csv (756.6 MB)... ✓ Loaded 3,575,768 rows
  4. pid_features_004.csv (698.3 MB)... ✓ Loaded 3,299,755 rows
  5. pid_features_005.csv (198.4 MB)... ✓ Loaded 939,037 rows
  6. pid_features_006.csv (881.0 MB)... ✓ Loaded 4,162,072 rows
  7. pid_features_007.csv (205.8 MB)... ✓ Loaded 973,803 rows
  8. pid_features_008.csv (117.8 MB)... ✓ Loaded 558,749 rows
  9. pid_features_009.csv (57.3 MB)... ✓ Loaded 271,973 rows

Combining 9 files...
✓ Combined shape: (20027673, 37)
  Total rows: 20,027,673
  Total columns: 37

Total tracks loaded from CSVs (all PDGs): 20,027,673

✓ Data loaded
Total tracks in dataset: 20,027,673

Track counts by momentum range:

NO DPG CUTS
--------------------------------------------

## Section 4.1: Compute TPC/TOF availability in given momentum ranges

In [7]:
def print_tpc_tof_availability_from_results(results, label="No DPG cuts", free_memory=False):
    """
    Memory-optimised computation of TPC/TOF availability.
    Includes clean, structured printing output.
    """
    features = results["features"]
    X_train = results["X_train"]
    X_test  = results["X_test"]

    pt_idx      = features.index("pt")
    has_tpc_idx = features.index("has_tpc")
    has_tof_idx = features.index("has_tof")

    # Extract columns ONCE (views, not copies)
    pt_tr      = X_train[:, pt_idx]
    has_tpc_tr = X_train[:, has_tpc_idx] > 0.5
    has_tof_tr = X_train[:, has_tof_idx] > 0.5

    pt_te      = X_test[:, pt_idx]
    has_tpc_te = X_test[:, has_tpc_idx] > 0.5
    has_tof_te = X_test[:, has_tof_idx] > 0.5

    ranges = [
        ("FULL", 0.0, np.inf),
        ("0–1",  0.0, 1.0),
        ("0.7–1.5", 0.7, 1.5),
        ("1–3",  1.0, 3.0),
        ("1–4",  1.0, 4.0),
        (">4",   4.0, np.inf),
    ]

    print("\n" + "=" * 80)
    print(f"TPC/TOF availability ({label})")
    print("=" * 80)

    # Header
    print(f"{'Range':<12} | {'Dataset':<6} | {'N':>10} | {'TPC %':>7} | {'TOF %':>7}")
    print("-" * 80)

    for name, p_min, p_max in ranges:
        # TRAIN
        mask_tr = (pt_tr >= p_min) & (pt_tr < p_max)
        n_tr = int(mask_tr.sum())

        if n_tr == 0:
            tpc_tr = tof_tr = 0.0
        else:
            tpc_tr = has_tpc_tr[mask_tr].mean() * 100
            tof_tr = has_tof_tr[mask_tr].mean() * 100

        # TEST
        mask_te = (pt_te >= p_min) & (pt_te < p_max)
        n_te = int(mask_te.sum())

        if n_te == 0:
            tpc_te = tof_te = 0.0
        else:
            tpc_te = has_tpc_te[mask_te].mean() * 100
            tof_te = has_tof_te[mask_te].mean() * 100

        unit = "" if name == "FULL" else " GeV/c"
        label_range = f"{name}{unit}"

        # Print rows
        print(f"{label_range:<12} | {'TRAIN':<6} | {n_tr:10d} | {tpc_tr:7.2f} | {tof_tr:7.2f}")
        print(f"{'':<12} | {'TEST':<6}  | {n_te:10d} | {tpc_te:7.2f} | {tof_te:7.2f}")
        print("-" * 80)

    # Optional: aggressively free memory
    if free_memory:
        del results, X_train, X_test
        del pt_tr, has_tpc_tr, has_tof_tr
        del pt_te, has_tpc_te, has_tof_te
        gc.collect()


# ============================================================================
# RUN BOTH CASES (MEMORY-SAFE SEQUENTIAL EXECUTION)
# ============================================================================

print("NO DPG CUTS")

momentum_range_no_dpg = {
    'key':  'full',
    'name': 'Full Spectrum (0.1-∞ GeV/c)',
    'min':  0.1,
    'max':  float('inf'),
    'apply_dpg_cuts': False,
}

results = preprocess_momentum_range(df, momentum_range_no_dpg)

print_tpc_tof_availability_from_results(
    results,
    label=f"{momentum_range_no_dpg['name']} (NO DPG cuts)",
    free_memory=True
)


print("\nWITH DPG CUTS")

momentum_range_dpg = {
    'key':  'full_DPG',
    'name': 'Full Spectrum (0.1-∞ GeV/c)',
    'min':  0.1,
    'max':  float('inf'),
    'apply_dpg_cuts': True,
}

results = preprocess_momentum_range(df, momentum_range_dpg)

print_tpc_tof_availability_from_results(
    results,
    label=f"{momentum_range_dpg['name']} (WITH DPG cuts)",
    free_memory=True
)


NO DPG CUTS

────────────────────────────────────────────────────────────────────────────────
Preprocessing: Full Spectrum (0.1-∞ GeV/c)  [key=full]
────────────────────────────────────────────────────────────────────────────────
  DPG cuts : NO
  Features : 25 (fixed)
  Groups   : 4 (tpc/tof/bayes/kinematics)

  After momentum filter: 20,027,670

  Bayesian PID status (raw data):
    REAL  (>0):  1,644,656 (8.21%)
    MISSING:     18,383,014 (91.79%)

  PDG filter: kept 17,600,795 / 20,027,670  (dropped 2,426,875)

  Train: 14,080,636  |  Test: 3,520,159
  ✓ Per-feature scalers fitted: 19
  ✓ Skipped (binary/sentinel):  ['bayes_prob_el_missing', 'bayes_prob_ka_missing', 'bayes_prob_pi_missing', 'bayes_prob_pr_missing', 'has_tof', 'has_tpc']
  Sample weights: real_bayes=3.0x  filled=1.0x

  Particle distribution (full range):
    Pion      : 12,236,938 ( 69.52%)
    Kaon      :  854,921 (  4.86%)
    Proton    : 2,125,809 ( 12.08%)
    Electron  : 2,383,127 ( 13.54%)

  Particle distri

## Section 4A: Train JAX_SimpleNN

In [ ]:
# ============================================================================
# SECTION 4A: TRAIN JAX_SIMPLENN MODEL (DETECTOR MODES)
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 4A: TRAINING JAX_SIMPLENN")
print(f"{'#'*80}\n")

# Train SimpleNN for all momentum ranges
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load SimpleNN
    force_training = FORCE_TRAINING['JAX_SimpleNN'][mr_key]
    results = train_model(
        model_type='JAX_SimpleNN',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    # ------------------------------------------------------------------
    # ADD DETECTOR MODES
    # ------------------------------------------------------------------
    X_test_full = preprocessing_data.get('X_test', preprocessing_data['X_test_scaled'])
    
    detector_modes = None
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')

        has_tpc = np.array(X_test_full[:, idx_tpc])
        has_tof = np.array(X_test_full[:, idx_tof])

        has_tpc_bin = (has_tpc >= 0.5).astype(int)
        has_tof_bin = (has_tof >= 0.5).astype(int)

        detector_modes = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
        
        print(f"✓ Added detector_modes for SimpleNN/{mr_key}: {len(detector_modes)} test samples")
        
    except Exception as e:
        print(f"Warning: could not build detector_modes for SimpleNN/{mr_key}: {e}")
        detector_modes = None
    
    # Store detector_modes in results
    if detector_modes is not None:
        results["detector_modes"] = detector_modes
    
    all_results_by_model_and_range[mr_key]['models']['JAX_SimpleNN'] = results

print(f"\n{'='*80}")
print("✓ SECTION 4A COMPLETE: JAX_SimpleNN trained/loaded + detector_modes added")
print(f"{'='*80}\n")

# Summary for SimpleNN
print("\nJAX_SimpleNN Summary:")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_SimpleNN']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")



################################################################################
SECTION 4A: TRAINING JAX_SIMPLENN
################################################################################


MOMENTUM RANGE: Full Spectrum (0.1-∞ GeV/c)


────────────────────────────────────────────────────────────────────────────────
Preprocessing: Full Spectrum (0.1-∞ GeV/c)  [key=full]
────────────────────────────────────────────────────────────────────────────────
  DPG cuts : NO
  Features : 25 (fixed)
  Groups   : 4 (tpc/tof/bayes/kinematics)

  After momentum filter: 20,027,670

  Bayesian PID status (raw data):
    REAL  (>0):  1,644,656 (8.21%)
    MISSING:     18,383,014 (91.79%)

  PDG filter: kept 17,600,795 / 20,027,670  (dropped 2,426,875)

  Train: 14,080,636  |  Test: 3,520,159
  ✓ Per-feature scalers fitted: 19
  ✓ Skipped (binary/sentinel):  ['bayes_prob_el_missing', 'bayes_prob_ka_missing', 'bayes_prob_pi_missing', 'bayes_prob_pr_missing', 'has_tof', 'has_tpc']
  Sample weights

## Section 4B: Train JAX_DNN

In [ ]:
# ============================================================================
# SECTION 4B: TRAIN JAX_DNN MODEL (DETECTOR MODES)
# ============================================================================

# Train DNN for all momentum ranges
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        # First model for this range - preprocess data
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        # Reuse existing preprocessing
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load DNN (unchanged)
    force_training = FORCE_TRAINING['JAX_DNN'][mr_key]
    
    results = train_model(
        model_type='JAX_DNN',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    # ------------------------------------------------------------------
    # ADD DETECTOR MODES
    # ------------------------------------------------------------------
    X_test_full = preprocessing_data.get('X_test', preprocessing_data['X_test_scaled'])
    
    detector_modes = None
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')

        has_tpc = np.array(X_test_full[:, idx_tpc])
        has_tof = np.array(X_test_full[:, idx_tof])

        has_tpc_bin = (has_tpc >= 0.5).astype(int)
        has_tof_bin = (has_tof >= 0.5).astype(int)

        detector_modes = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
        
        print(f"✓ Added detector_modes for DNN/{mr_key}: {len(detector_modes)} test samples")
        
    except Exception as e:
        print(f"Warning: could not build detector_modes for DNN/{mr_key}: {e}")
        detector_modes = None
    
    # Store detector_modes in results
    if detector_modes is not None:
        results["detector_modes"] = detector_modes
    
    all_results_by_model_and_range[mr_key]['models']['JAX_DNN'] = results

print(f"\n{'='*80}")
print("✓ SECTION 4B COMPLETE: JAX_DNN trained/loaded + detector_modes added")
print(f"{'='*80}\n")

# Summary for DNN
print("\nJAX_DNN Summary:")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_DNN']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")


## Section 4C: Train JAX_FSE_Attention_Detector_Aware (models merged)

In [ ]:
# ============================================================================
# SECTION 4C: TRAIN JAX_FSE_ATTENTION_DETECTORAWARE (models merged + DETECTOR MODES)
# ============================================================================

print(f"\n{'*'*80}")
print("JAX_FSE_ATTENTION_DETECTORAWARE (models merged)")
print(f"{'*'*80}\n")

"""
Baseline FSE_Attention is already learning detector information implicitly.
    
In this dataset, attention mechanisms are sufficient to capture detector-dependent behaviour without explicit conditioning.
    
Detector-aware conditioning does not significantly outperform implicit attention modelling, 
suggesting that detector information is largely encoded in input features.
"""

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    
    print(f"\n{'='*80}")
    print(f"MOMENTUM RANGE: {momentum_range['name']}")
    print(f"{'='*80}\n")
    
    # Get or create preprocessing data
    if mr_key not in all_results_by_model_and_range:
        # First model for this range - preprocess data
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data,
            'models': {}
        }
    else:
        # Reuse existing preprocessing
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']
    
    # Train/load FSE+Attention
    force_training = FORCE_TRAINING['JAX_FSE_Attention'][mr_key]
    
    results = train_model(
        model_type='JAX_FSE_Attention',
        momentum_range=momentum_range,
        preprocessing_data=preprocessing_data,
        force_training=force_training
    )
    
    # ------------------------------------------------------------------
    # ADD DETECTOR MODES
    # ------------------------------------------------------------------
    X_test_full = preprocessing_data.get('X_test', preprocessing_data['X_test_scaled'])
    
    detector_modes = None
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')

        has_tpc = np.array(X_test_full[:, idx_tpc])
        has_tof = np.array(X_test_full[:, idx_tof])

        has_tpc_bin = (has_tpc >= 0.5).astype(int)
        has_tof_bin = (has_tof >= 0.5).astype(int)

        detector_modes = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
        
        print(f"✓ Added detector_modes for FSE_Attention/{mr_key}: {len(detector_modes)} test samples")
        
    except Exception as e:
        print(f"Warning: could not build detector_modes for FSE_Attention/{mr_key}: {e}")
        detector_modes = None
    
    # Store detector_modes in results
    if detector_modes is not None:
        results["detector_modes"] = detector_modes
    
    all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention'] = results

print(f"\n{'='*80}")
print("✓ SECTION 4C COMPLETE: JAX_FSE_Attention (models merged) + detector_modes added")
print(f"{'='*80}\n")

# Summary for FSE+Attention
print("\nJAX_FSE_Attention (models merged) Summary:")
for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    results = all_results_by_model_and_range[mr_key]['models']['JAX_FSE_Attention']
    print(f"  {momentum_range['name']:30s}: Test Acc = {results['test_acc']:.4f}")


## Section 4E: Train LightGBM

In [ ]:
# ============================================================================
# SECTION 4E: LIGHTGBM MODEL
# ============================================================================

print("\n" + "="*80)
print("SECTION 4E: LIGHTGBM TRAINING")
print("="*80)

lgbm_results = {}
params = HYPERPARAMETERS['LightGBM']

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']

    print(f"\n{'*'*80}")
    print(f"LightGBM — {momentum_range['name']}")
    print(f"{'*'*80}")

    if mr_key not in all_results_by_model_and_range:
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data, 'models': {}}
    else:
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']

    X_train = preprocessing_data['X_train_scaled']
    X_test  = preprocessing_data['X_test_scaled']
    y_train = preprocessing_data['y_train']
    y_test  = preprocessing_data['y_test']
    X_test_raw = preprocessing_data.get('X_test', X_test)

    model_path     = f"/kaggle/working/trained_models/{mr_key}_LightGBM.pkl"
    force_training = FORCE_TRAINING['LightGBM'][mr_key]

    if os.path.exists(model_path) and not force_training:
        print(f"✓ Loaded from: {model_path}")
        with open(model_path, "rb") as f:
            lgb_model = pickle.load(f)
    else:
        print("Training LightGBM...")
        lgb_model = lgb.LGBMClassifier(
            objective='multiclass',
            num_class=NUM_CLASSES,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            num_leaves=params['num_leaves'],
            max_depth=params['max_depth'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            random_state=params['random_state'],
            n_jobs=params['n_jobs'],
        )
        lgb_model.fit(X_train, y_train)
        with open(model_path, "wb") as f:
            pickle.dump(lgb_model, f)
        print(f"✓ Model object saved to: {model_path}")

    y_pred_test = lgb_model.predict(X_test)
    test_probs  = None
    try:
        test_probs = lgb_model.predict_proba(X_test)
    except Exception:
        pass

    test_acc = float(accuracy_score(y_test, y_pred_test))
    print(f"Test Accuracy: {test_acc:.4f}")

    # ── detector modes ────────────────────────────────────────────────────
    detector_modes = None
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')
        has_tpc_bin = (np.array(X_test_raw[:, idx_tpc]) >= 0.5).astype(int)
        has_tof_bin = (np.array(X_test_raw[:, idx_tof]) >= 0.5).astype(int)
        detector_modes = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
    except Exception as ex:
        print(f"Warning: could not build detector_modes for LightGBM/{mr_key}: {ex}")

    # ── results dict with full inference contract ─────────────────────────
    result = {
        'model':           lgb_model,
        'test_acc':        test_acc,
        'y_pred_test':     y_pred_test,
        'y_test':          y_test,
        'test_probs':      test_probs,
        # inference contract
        'features':        list(TRAINING_FEATURES),
        'feature_scalers': preprocessing_data.get('feature_scalers', {}),
        'num_groups':      len(DETECTOR_GROUPS),
        'group_names':     list(DETECTOR_GROUPS.keys()),
        'skip_scaling':    list(SKIP_SCALING),
    }
    if detector_modes is not None:
        result['detector_modes'] = detector_modes

    # save full result dict (includes model object + contract)
    result_path = f"/kaggle/working/trained_models/{mr_key}_LightGBM_result.pkl"
    os.makedirs(os.path.dirname(result_path), exist_ok=True)
    with open(result_path, 'wb') as f:
        pickle.dump(result, f)
    print(f"✓ Result dict saved to: {result_path}")

    lgbm_results[mr_key] = result
    all_results_by_model_and_range[mr_key]['models']['LightGBM'] = result

print("\n" + "="*80)
print("✓ SECTION 4E COMPLETE: LightGBM trained for all ranges")
print("="*80 + "\n")

print("\nLightGBM Summary:")
for mr in MOMENTUM_RANGES:
    r = all_results_by_model_and_range[mr['key']]['models']['LightGBM']
    print(f"  {mr['name']:45s}: Test Acc = {r['test_acc']:.4f}")
    

## Section 4F: Train XGBoost

In [ ]:
# ============================================================================
# SECTION 4F: XGBOOST MODEL
# ============================================================================

print("\n" + "="*80)
print("SECTION 4F: XGBOOST TRAINING")
print("="*80)

xgb_results = {}
params = HYPERPARAMETERS['XGBoost']

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']

    print(f"\n{'*'*80}")
    print(f"XGBoost — {momentum_range['name']}")
    print(f"{'*'*80}")

    if mr_key not in all_results_by_model_and_range:
        preprocessing_data = preprocess_momentum_range(df, momentum_range)
        all_results_by_model_and_range[mr_key] = {
            'preprocessing': preprocessing_data, 'models': {}}
    else:
        preprocessing_data = all_results_by_model_and_range[mr_key]['preprocessing']

    X_train = preprocessing_data['X_train_scaled']
    X_test  = preprocessing_data['X_test_scaled']
    y_train = preprocessing_data['y_train']
    y_test  = preprocessing_data['y_test']
    X_test_raw = preprocessing_data.get('X_test', X_test)

    model_path     = f"/kaggle/working/trained_models/{mr_key}_XGBoost.pkl"
    force_training = FORCE_TRAINING['XGBoost'][mr_key]

    if os.path.exists(model_path) and not force_training:
        print(f"✓ Loaded from: {model_path}")
        with open(model_path, "rb") as f:
            xgb_model = pickle.load(f)
    else:
        print("Training XGBoost...")
        xgb_model = xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=NUM_CLASSES,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params['max_depth'],
            subsample=params['subsample'],
            colsample_bytree=params['colsample_bytree'],
            tree_method=params['tree_method'],
            random_state=params['random_state'],
            n_jobs=params['n_jobs'],
            verbosity=0,
        )
        xgb_model.fit(X_train, y_train)
        with open(model_path, "wb") as f:
            pickle.dump(xgb_model, f)
        print(f"✓ Model object saved to: {model_path}")

    y_pred_test = xgb_model.predict(X_test)
    test_probs  = None
    try:
        test_probs = xgb_model.predict_proba(X_test)
    except Exception:
        pass

    test_acc = float(accuracy_score(y_test, y_pred_test))
    print(f"Test Accuracy: {test_acc:.4f}")

    # ── detector modes ────────────────────────────────────────────────────
    detector_modes = None
    try:
        idx_tpc = TRAINING_FEATURES.index('has_tpc')
        idx_tof = TRAINING_FEATURES.index('has_tof')
        has_tpc_bin = (np.array(X_test_raw[:, idx_tpc]) >= 0.5).astype(int)
        has_tof_bin = (np.array(X_test_raw[:, idx_tof]) >= 0.5).astype(int)
        detector_modes = np.full(len(has_tpc_bin), 'NONE', dtype=object)
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 0)] = 'TPC_ONLY'
        detector_modes[(has_tpc_bin == 0) & (has_tof_bin == 1)] = 'TOF_ONLY'
        detector_modes[(has_tpc_bin == 1) & (has_tof_bin == 1)] = 'TPC_TOF'
    except Exception as ex:
        print(f"Warning: could not build detector_modes for XGBoost/{mr_key}: {ex}")

    # ── results dict with full inference contract ─────────────────────────
    result = {
        'model':           xgb_model,
        'test_acc':        test_acc,
        'y_pred_test':     y_pred_test,
        'y_test':          y_test,
        'test_probs':      test_probs,
        # inference contract
        'features':        list(TRAINING_FEATURES),
        'feature_scalers': preprocessing_data.get('feature_scalers', {}),
        'num_groups':      len(DETECTOR_GROUPS),
        'group_names':     list(DETECTOR_GROUPS.keys()),
        'skip_scaling':    list(SKIP_SCALING),
    }
    if detector_modes is not None:
        result['detector_modes'] = detector_modes

    # save full result dict
    result_path = f"/kaggle/working/trained_models/{mr_key}_XGBoost_result.pkl"
    os.makedirs(os.path.dirname(result_path), exist_ok=True)
    with open(result_path, 'wb') as f:
        pickle.dump(result, f)
    print(f"✓ Result dict saved to: {result_path}")

    xgb_results[mr_key] = result
    all_results_by_model_and_range[mr_key]['models']['XGBoost'] = result

print("\n" + "="*80)
print("✓ SECTION 4F COMPLETE: XGBoost trained for all ranges")
print("="*80 + "\n")

print("\nXGBoost Summary:")
for mr in MOMENTUM_RANGES:
    r = all_results_by_model_and_range[mr['key']]['models']['XGBoost']
    print(f"  {mr['name']:45s}: Test Acc = {r['test_acc']:.4f}")
    

# Section 5: Statistics & Plots

## Section 5A: No-cuts vs DPG-cuts Summary

In [ ]:
# =============================================================================
# SECTION 5A: NO-CUTS vs DPG-CUTS SUMMARY
# =============================================================================

print("\n" + "="*80)
print("SECTION 5A: NO-CUTS vs DPG-CUTS SUMMARY")
print("="*80 + "\n")

# Build a lookup from momentum-range key -> dict
momentum_by_key = {mr['key']: mr for mr in MOMENTUM_RANGES}

# Find (baseline_key, dpg_key) pairs based on *_DPG naming
pairs = []
for key in momentum_by_key:
    if key.endswith("_DPG"):
        base_key = key[:-4]  # strip "_DPG"
        if base_key in momentum_by_key:
            pairs.append((base_key, key))

if not pairs:
    print("No DPG momentum ranges found (expected keys like 'full_DPG').")
else:
    for base_key, dpg_key in pairs:
        # Guard against missing results if some ranges were not trained
        if (base_key not in all_results_by_model_and_range or
                dpg_key not in all_results_by_model_and_range):
            print(f"Skipping '{base_key}' – missing baseline or DPG results in all_results_by_model_and_range.")
            continue

        base_name = momentum_by_key[base_key]['name']
        dpg_name  = momentum_by_key[dpg_key]['name']

        print("\n" + "-"*80)
        print(f"Momentum range: {base_name}")
        print("-"*80)

        # ----------------------------
        # Track-count comparison
        # ----------------------------
        base_prep = all_results_by_model_and_range[base_key]['preprocessing']
        dpg_prep  = all_results_by_model_and_range[dpg_key]['preprocessing']

        base_n_train = len(base_prep['X_train_scaled'])
        base_n_test  = len(base_prep['X_test_scaled'])
        dpg_n_train  = len(dpg_prep['X_train_scaled'])
        dpg_n_test   = len(dpg_prep['X_test_scaled'])

        # Reductions (relative to baseline)
        train_reduction = 100.0 * (1.0 - dpg_n_train / max(base_n_train, 1))
        test_reduction  = 100.0 * (1.0 - dpg_n_test / max(base_n_test, 1))

        print(f"  Tracks (train/test):")
        print(f"    no-cuts : {base_n_train:,} / {base_n_test:,}")
        print(f"    DPG-cuts: {dpg_n_train:,} / {dpg_n_test:,}")
        print(f"    Reduction (train): {train_reduction:6.2f}%")
        print(f"    Reduction (test) : {test_reduction:6.2f}%")

        # ----------------------------
        # Per-model test accuracy comparison
        # ----------------------------
        print("\n  Per-model test accuracy:")
        for model_type in MODEL_TYPES:
            base_res = all_results_by_model_and_range[base_key]['models'].get(model_type)
            dpg_res  = all_results_by_model_and_range[dpg_key]['models'].get(model_type)

            if base_res is None or dpg_res is None:
                print(f"    {model_type:18s}: (missing results)")
                continue

            base_acc = base_res['test_acc']
            dpg_acc  = dpg_res['test_acc']
            delta    = dpg_acc - base_acc

            print(
                f"    {model_type:18s}: "
                f"no-cuts = {base_acc:.4f}, "
                f"DPG-cuts = {dpg_acc:.4f}, "
                f"Δ = {delta:+.4f}"
            )

print("\n" + "="*80)
print("END OF SECTION 5A")
print("="*80 + "\n")


## Section 5B: Comparison & Performance

In [ ]:
# ============================================================================
# SECTION 5B: COMPARISON & PERFORMANCE (WITH / WITHOUT DPG CUTS)
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 5: COMPARISON & PERFORMANCE (WITH / WITHOUT DPG CUTS)")
print(f"{'#'*80}\n")

# ---------------------------------------------------------------------------
# Helper: split momentum ranges into no-cuts vs DPG-cuts
# ---------------------------------------------------------------------------
momentum_by_key = {mr['key']: mr for mr in MOMENTUM_RANGES}

base_keys = [mr['key'] for mr in MOMENTUM_RANGES
             if not mr.get('apply_dpg_cuts', False)]
dpg_keys  = [mr['key'] for mr in MOMENTUM_RANGES
             if mr.get('apply_dpg_cuts', False)]

print("Non-DPG momentum ranges:", base_keys)
print("DPG momentum ranges    :", dpg_keys)

# Optional: display names for models (override safely if already defined)
model_display_names = {
    'JAX_SimpleNN':      'SimpleNN',
    'JAX_DNN':           'DNN',
    'JAX_FSE_Attention': 'FSE+Attention',
    'LightGBM':          'LightGBM',
    'XGBoost':           'XGBoost',
}

# ============================================================================
# PART 1: TEST ACCURACY COMPARISON – NO DPG CUTS
# ============================================================================

print(f"\n{'='*80}")
print("TEST ACCURACY COMPARISON (All Models, All Momentum Ranges – NO DPG CUTS)")
print(f"{'='*80}\n")

if base_keys:
    fig, axes = plt.subplots(
        1, len(base_keys),
        figsize=(5 * len(base_keys), 5),
        squeeze=False
    )

    for idx, mr_key in enumerate(base_keys):
        momentum_range = momentum_by_key[mr_key]
        ax = axes[0, idx]

        x_pos = np.arange(len(MODEL_TYPES))
        test_accs = []
        colors = []

        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        for model_type in MODEL_TYPES:
            if model_type in models_dict:
                results = models_dict[model_type]
                test_accs.append(results['test_acc'])
                colors.append(model_colors.get(model_type, '#999999'))
            else:
                test_accs.append(0.0)
                colors.append('#CCCCCC')

        bars = ax.bar(x_pos, test_accs,
                      color=colors, alpha=0.8,
                      edgecolor='black', linewidth=1.5)

        # Add value labels
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width() / 2., height + 0.01,
                        f'{height:.3f}', ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

        ax.set_ylabel('Test Accuracy', fontsize=11, fontweight='bold')
        ax.set_title(momentum_range['name'], fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            [model_display_names.get(m, m) for m in MODEL_TYPES],
            rotation=45, ha='right', fontsize=9
        )
        ax.set_ylim([0, 1.0])
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Test Accuracy Comparison (All Models, All Momentum Ranges – NO DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

    print("✓ Accuracy comparison plots generated (NO DPG cuts)")
else:
    print("No non-DPG momentum ranges found; skipping Part 1.")

# ============================================================================
# PART 2: TEST ACCURACY COMPARISON – WITH DPG CUTS
# ============================================================================

print(f"\n{'='*80}")
print("TEST ACCURACY COMPARISON (All Models, All Momentum Ranges – WITH DPG CUTS)")
print(f"{'='*80}\n")

if dpg_keys:
    fig, axes = plt.subplots(
        1, len(dpg_keys),
        figsize=(5 * len(dpg_keys), 5),
        squeeze=False
    )

    for idx, mr_key in enumerate(dpg_keys):
        momentum_range = momentum_by_key[mr_key]
        ax = axes[0, idx]

        x_pos = np.arange(len(MODEL_TYPES))
        test_accs = []
        colors = []

        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        for model_type in MODEL_TYPES:
            if model_type in models_dict:
                results = models_dict[model_type]
                test_accs.append(results['test_acc'])
                colors.append(model_colors.get(model_type, '#999999'))
            else:
                test_accs.append(0.0)
                colors.append('#CCCCCC')

        bars = ax.bar(x_pos, test_accs,
                      color=colors, alpha=0.8,
                      edgecolor='black', linewidth=1.5)

        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width() / 2., height + 0.01,
                        f'{height:.3f}', ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

        ax.set_ylabel('Test Accuracy', fontsize=11, fontweight='bold')
        ax.set_title(momentum_range['name'], fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            [model_display_names.get(m, m) for m in MODEL_TYPES],
            rotation=45, ha='right', fontsize=9
        )
        ax.set_ylim([0, 1.0])
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Test Accuracy Comparison (All Models, All Momentum Ranges – WITH DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

    print("✓ Accuracy comparison plots generated (WITH DPG cuts)")
else:
    print("No DPG momentum ranges found; skipping Part 2.")

# ============================================================================
# PART 3: BEST-MODEL CONFUSION MATRICES – NO CUTS vs DPG CUTS
# ============================================================================

print(f"\n{'='*80}")
print("BEST MODEL CONFUSION MATRICES – NO CUTS vs WITH DPG CUTS")
print(f"{'='*80}\n")

from sklearn.metrics import confusion_matrix

# Helper to plot best-model confusion matrices for a set of keys
def plot_best_confusions_for_keys(mr_keys, title_suffix):
    if not mr_keys:
        print(f"No momentum ranges for {title_suffix}; skipping.")
        return

    fig, axes = plt.subplots(
        2, len(mr_keys),
        figsize=(5 * len(mr_keys), 10),
        squeeze=False
    )

    for idx, mr_key in enumerate(mr_keys):
        momentum_range = momentum_by_key[mr_key]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        if not models_dict:
            print(f"No models found for {momentum_range['name']} ({mr_key}); skipping.")
            continue

        print(f"\n{momentum_range['name']} [{title_suffix}]")
        print("─" * 70)

        # Select best model by test accuracy
        best_model = None
        best_acc = -1.0

        for model_type in MODEL_TYPES:
            if model_type in models_dict:
                acc = models_dict[model_type]['test_acc']
                print(f"  {model_display_names.get(model_type, model_type):20s}: {acc:.4f}")
                if acc > best_acc:
                    best_acc = acc
                    best_model = model_type

        if best_model is None:
            print("  No valid models for this range; skipping confusion matrices.")
            continue

        print(f"\n✓ Best Model: {model_display_names.get(best_model, best_model)} ({best_acc:.4f})\n")

        y_test = np.array(models_dict[best_model]['y_test'])
        y_pred = np.array(models_dict[best_model]['y_pred_test'])

        cm = confusion_matrix(y_test, y_pred)
        cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

        # Counts matrix
        ax_counts = axes[0, idx]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax_counts,
                    xticklabels=PARTICLE_NAMES, yticklabels=PARTICLE_NAMES,
                    cbar_kws={'label': 'Count'}, vmin=0)
        ax_counts.set_ylabel('True Label', fontsize=10, fontweight='bold')
        ax_counts.set_xlabel('Predicted Label', fontsize=10, fontweight='bold')
        ax_counts.set_title(
            f"{model_display_names.get(best_model, best_model)}\n"
            f"{momentum_range['name']} ({title_suffix}, Counts)",
            fontsize=11, fontweight='bold'
        )

        # Normalised matrix
        ax_norm = axes[1, idx]
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax_norm,
                    xticklabels=PARTICLE_NAMES, yticklabels=PARTICLE_NAMES,
                    cbar_kws={'label': 'Recall'}, vmin=0, vmax=1)
        ax_norm.set_ylabel('True Label', fontsize=10, fontweight='bold')
        ax_norm.set_xlabel('Predicted Label', fontsize=10, fontweight='bold')
        ax_norm.set_title(
            f"{model_display_names.get(best_model, best_model)}\n"
            f"{momentum_range['name']} ({title_suffix}, Normalised)",
            fontsize=11, fontweight='bold'
        )

    plt.suptitle(
        f"Best Model Confusion Matrices ({title_suffix})\n"
        f"Top: Counts, Bottom: Normalised by True Class",
        fontsize=13, fontweight='bold', y=0.995
    )
    plt.tight_layout()
    plt.show()

# Confusions without cuts
plot_best_confusions_for_keys(base_keys, "NO DPG CUTS")

# Confusions with DPG cuts
plot_best_confusions_for_keys(dpg_keys, "WITH DPG CUTS")

print("\n" + "="*80)
print("✓ Confusion matrices generated for best models (NO vs WITH DPG cuts)")
print("="*80)

# ============================================================================
# PART 4: PERFORMANCE RANKING – NO CUTS vs DPG CUTS vs COMBINED
# ============================================================================

print(f"\n{'='*80}")
print("PERFORMANCE RANKING – All Models, All Ranges")
print(f"{'='*80}\n")

def build_ranking_for_keys(mr_keys, label):
    ranking_data = []

    for mr_key in mr_keys:
        momentum_range = momentum_by_key[mr_key]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        for model_type in MODEL_TYPES:
            if model_type in models_dict:
                results = models_dict[model_type]
                ranking_data.append({
                    'Model': model_display_names.get(model_type, model_type),
                    'Momentum Range': momentum_range['name'],
                    'Set': label,
                    'Test Acc': results['test_acc'],
                    'Train Time (s)': results.get('training_time', 0.0),
                })

    if not ranking_data:
        return None

    df = pd.DataFrame(ranking_data).sort_values('Test Acc', ascending=False).reset_index(drop=True)
    df.insert(0, 'Rank', np.arange(1, len(df) + 1))
    return df

# No DPG cuts
ranking_no_dpg = build_ranking_for_keys(base_keys, 'NO DPG CUTS')
if ranking_no_dpg is not None:
    print("PERFORMANCE RANKING (NO DPG CUTS):\n")
    print(ranking_no_dpg.to_string(index=False))
    print()
else:
    print("No data for NO DPG CUTS ranking.\n")

# With DPG cuts
ranking_dpg = build_ranking_for_keys(dpg_keys, 'WITH DPG CUTS')
if ranking_dpg is not None:
    print("PERFORMANCE RANKING (WITH DPG CUTS):\n")
    print(ranking_dpg.to_string(index=False))
    print()
else:
    print("No data for WITH DPG CUTS ranking.\n")

# Combined
if ranking_no_dpg is not None or ranking_dpg is not None:
    combined_parts = []
    if ranking_no_dpg is not None:
        combined_parts.append(ranking_no_dpg.drop(columns=['Rank']))
    if ranking_dpg is not None:
        combined_parts.append(ranking_dpg.drop(columns=['Rank']))

    ranking_all = pd.concat(combined_parts, ignore_index=True)
    ranking_all = ranking_all.sort_values('Test Acc', ascending=False).reset_index(drop=True)
    ranking_all.insert(0, 'Rank', np.arange(1, len(ranking_all) + 1))

    print("PERFORMANCE RANKING (NO DPG CUTS + WITH DPG CUTS COMBINED):\n")
    print(ranking_all.to_string(index=False))
    print()
else:
    print("No data for combined ranking.\n")

print(f"{'='*80}")
print("✓ SECTION 5B COMPLETE: Comparisons and rankings with/without DPG cuts")
print(f"{'='*80}\n")


## Section 5C: ROC / AUC Analysis

In [ ]:
# ============================================================================
# SECTION 5C: ROC / AUC ANALYSIS WITH & WITHOUT DPG CUTS
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 5C: ROC / AUC ANALYSIS (WITH & WITHOUT DPG CUTS)")
print(f"{'#'*80}\n")

# ---------------------------------------------------------------------------
# Helper: safe macro AUC (reusable)
# ---------------------------------------------------------------------------
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def calculate_macro_auc_safe(y_test, y_test_probs):
    """
    Safely calculate macro AUC from test probabilities.
    Returns NaN if calculation fails.
    """
    try:
        y_test = np.array(y_test)
        y_test_probs = np.array(y_test_probs)
        y_test_bin = label_binarize(y_test, classes=np.arange(NUM_CLASSES))
        auc_scores = []
        for i in range(NUM_CLASSES):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_test_probs[:, i])
            auc_score = auc(fpr, tpr)
            auc_scores.append(auc_score)
        return float(np.mean(auc_scores))
    except Exception:
        return np.nan

# ---------------------------------------------------------------------------
# Split momentum ranges: no DPG vs DPG
# ---------------------------------------------------------------------------
momentum_by_key = {mr['key']: mr for mr in MOMENTUM_RANGES}
base_keys = [mr['key'] for mr in MOMENTUM_RANGES
             if not mr.get('apply_dpg_cuts', False)]
dpg_keys  = [mr['key'] for mr in MOMENTUM_RANGES
             if mr.get('apply_dpg_cuts', False)]

print("Non-DPG momentum ranges:", base_keys)
print("DPG momentum ranges    :", dpg_keys)

# Display names for plots
model_display_names = {
    'JAX_SimpleNN':      'SimpleNN',
    'JAX_DNN':           'DNN',
    'JAX_FSE_Attention': 'FSE+Attention',
    'LightGBM':          'LightGBM',
    'XGBoost':           'XGBoost',
}

# ============================================================================
# PART 1: ROC CURVES – MACRO AUC (NO DPG vs WITH DPG)
# ============================================================================

print(f"\n{'='*80}")
print("ROC CURVES: MACRO AUC (All Models, All Momentum Ranges – NO DPG CUTS)")
print(f"{'='*80}\n")

macro_auc_no_dpg = {}   # {mr_key: {model_type: macro_auc}}

if base_keys:
    fig, axes = plt.subplots(
        1, len(base_keys),
        figsize=(6 * len(base_keys), 6),
        squeeze=False
    )

    for idx, mr_key in enumerate(base_keys):
        mr = momentum_by_key[mr_key]
        ax = axes[0, idx]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})
        macro_auc_no_dpg[mr_key] = {}

        if not models_dict:
            ax.set_title(f"{mr['name']} (no models)")
            ax.axis('off')
            continue

        # Chance line
        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.2, label='Chance')

        for model_type in MODEL_TYPES:
            if model_type not in models_dict:
                continue

            results = models_dict[model_type]
            try:
                y_test = np.array(results['y_test'])
                y_probs = np.array(results['test_probs'])
                y_bin = label_binarize(y_test, classes=np.arange(NUM_CLASSES))

                # Per-class ROC
                fpr_all, tpr_all, auc_scores = [], [], []
                for i in range(NUM_CLASSES):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
                    auc_i = auc(fpr, tpr)
                    fpr_all.append(fpr)
                    tpr_all.append(tpr)
                    auc_scores.append(auc_i)

                macro_auc = float(np.mean(auc_scores))
                macro_auc_no_dpg[mr_key][model_type] = macro_auc

                # Interpolate to common grid for smooth macro curve
                all_fpr = np.unique(np.concatenate(fpr_all))
                mean_tpr = np.zeros_like(all_fpr)
                for i in range(NUM_CLASSES):
                    mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
                mean_tpr /= NUM_CLASSES

                ax.plot(all_fpr, mean_tpr,
                        label=f"{model_display_names.get(model_type, model_type)} (AUC={macro_auc:.3f})",
                        color=model_colors.get(model_type, '#999999'),
                        lw=2.0, alpha=0.85)
            except Exception as e:
                print(f"Warning: ROC failed for {model_type} / {mr_key}: {str(e)[:80]}")

        ax.set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
        ax.set_title(mr['name'], fontsize=11, fontweight='bold')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('ROC Curves: Macro AUC (NO DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("No non-DPG momentum ranges; skipping ROC (NO DPG CUTS).")

print(f"\n{'='*80}")
print("ROC CURVES: MACRO AUC (All Models, All Momentum Ranges – WITH DPG CUTS)")
print(f"{'='*80}\n")

macro_auc_dpg = {}      # {mr_key: {model_type: macro_auc}}

if dpg_keys:
    fig, axes = plt.subplots(
        1, len(dpg_keys),
        figsize=(6 * len(dpg_keys), 6),
        squeeze=False
    )

    for idx, mr_key in enumerate(dpg_keys):
        mr = momentum_by_key[mr_key]
        ax = axes[0, idx]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})
        macro_auc_dpg[mr_key] = {}

        if not models_dict:
            ax.set_title(f"{mr['name']} (no models)")
            ax.axis('off')
            continue

        ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.2, label='Chance')

        for model_type in MODEL_TYPES:
            if model_type not in models_dict:
                continue

            results = models_dict[model_type]
            try:
                y_test = np.array(results['y_test'])
                y_probs = np.array(results['test_probs'])
                y_bin = label_binarize(y_test, classes=np.arange(NUM_CLASSES))

                fpr_all, tpr_all, auc_scores = [], [], []
                for i in range(NUM_CLASSES):
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
                    auc_i = auc(fpr, tpr)
                    fpr_all.append(fpr)
                    tpr_all.append(tpr)
                    auc_scores.append(auc_i)

                macro_auc = float(np.mean(auc_scores))
                macro_auc_dpg[mr_key][model_type] = macro_auc

                all_fpr = np.unique(np.concatenate(fpr_all))
                mean_tpr = np.zeros_like(all_fpr)
                for i in range(NUM_CLASSES):
                    mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
                mean_tpr /= NUM_CLASSES

                ax.plot(all_fpr, mean_tpr,
                        label=f"{model_display_names.get(model_type, model_type)} (AUC={macro_auc:.3f})",
                        color=model_colors.get(model_type, '#999999'),
                        lw=2.0, alpha=0.85)
            except Exception as e:
                print(f"Warning: ROC failed for {model_type} / {mr_key}: {str(e)[:80]}")

        ax.set_xlabel('False Positive Rate', fontsize=10, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=10, fontweight='bold')
        ax.set_title(mr['name'], fontsize=11, fontweight='bold')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('ROC Curves: Macro AUC (WITH DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
else:
    print("No DPG momentum ranges; skipping ROC (WITH DPG CUTS).")

# ============================================================================
# PART 2: ONE-vs-REST ROC CURVES (NO DPG vs WITH DPG)
# ============================================================================

print(f"\n{'='*80}")
print("ONE-VS-REST ROC CURVES (All Models, All Ranges – NO DPG CUTS)")
print(f"{'='*80}\n")

if base_keys:
    fig, axes = plt.subplots(
        len(base_keys), len(PARTICLE_NAMES),
        figsize=(4 * len(PARTICLE_NAMES), 4 * len(base_keys)),
        squeeze=False
    )

    for r_idx, mr_key in enumerate(base_keys):
        mr = momentum_by_key[mr_key]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        for p_idx, particle_name in enumerate(PARTICLE_NAMES):
            ax = axes[r_idx, p_idx]
            ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.2)

            for model_type in MODEL_TYPES:
                if model_type not in models_dict:
                    continue
                try:
                    results = models_dict[model_type]
                    y_test = np.array(results['y_test'])
                    y_probs = np.array(results['test_probs'])

                    y_bin = (y_test == p_idx).astype(int)
                    fpr, tpr, _ = roc_curve(y_bin, y_probs[:, p_idx])
                    roc_auc_val = auc(fpr, tpr)

                    ax.plot(fpr, tpr,
                            label=f"{model_display_names.get(model_type, model_type)} ({roc_auc_val:.3f})",
                            color=model_colors.get(model_type, '#999999'),
                            lw=2.0, alpha=0.85)
                except Exception:
                    pass

            ax.set_xlabel('False Positive Rate', fontsize=9, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=9, fontweight='bold')
            ax.set_title(f"{particle_name} ({mr['name']})", fontsize=10, fontweight='bold')
            ax.set_xlim([0, 1])
            ax.set_ylim([0, 1])
            ax.legend(loc='lower right', fontsize=7)
            ax.grid(alpha=0.3)

    plt.suptitle('One-vs-Rest ROC (NO DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
else:
    print("No non-DPG ranges; skipping one-vs-rest (NO DPG CUTS).")

print(f"\n{'='*80}")
print("ONE-VS-REST ROC CURVES (All Models, All Ranges – WITH DPG CUTS)")
print(f"{'='*80}\n")

if dpg_keys:
    fig, axes = plt.subplots(
        len(dpg_keys), len(PARTICLE_NAMES),
        figsize=(4 * len(PARTICLE_NAMES), 4 * len(dpg_keys)),
        squeeze=False
    )

    for r_idx, mr_key in enumerate(dpg_keys):
        mr = momentum_by_key[mr_key]
        models_dict = all_results_by_model_and_range.get(mr_key, {}).get('models', {})

        for p_idx, particle_name in enumerate(PARTICLE_NAMES):
            ax = axes[r_idx, p_idx]
            ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.2)

            for model_type in MODEL_TYPES:
                if model_type not in models_dict:
                    continue
                try:
                    results = models_dict[model_type]
                    y_test = np.array(results['y_test'])
                    y_probs = np.array(results['test_probs'])

                    y_bin = (y_test == p_idx).astype(int)
                    fpr, tpr, _ = roc_curve(y_bin, y_probs[:, p_idx])
                    roc_auc_val = auc(fpr, tpr)

                    ax.plot(fpr, tpr,
                            label=f"{model_display_names.get(model_type, model_type)} ({roc_auc_val:.3f})",
                            color=model_colors.get(model_type, '#999999'),
                            lw=2.0, alpha=0.85)
                except Exception:
                    pass

            ax.set_xlabel('False Positive Rate', fontsize=9, fontweight='bold')
            ax.set_ylabel('True Positive Rate', fontsize=9, fontweight='bold')
            ax.set_title(f"{particle_name} ({mr['name']})", fontsize=10, fontweight='bold')
            ax.set_xlim([0, 1])
            ax.set_ylim([0, 1])
            ax.legend(loc='lower right', fontsize=7)
            ax.grid(alpha=0.3)

    plt.suptitle('One-vs-Rest ROC (WITH DPG CUTS)',
                 fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()
else:
    print("No DPG ranges; skipping one-vs-rest (WITH DPG CUTS).")

# ============================================================================
# PART 3: MACRO AUC SUMMARY TABLES & RANKINGS
# ============================================================================

print(f"\n{'='*80}")
print("MACRO AUC SUMMARY TABLES & RANKINGS")
print(f"{'='*80}\n")

# Summary tables (macro AUC, all models, all ranges)

def build_macro_auc_rows(macro_dict, label):
    rows = []
    for mr_key, models_dict in macro_dict.items():
        mr = momentum_by_key[mr_key]
        for model_type, auc_val in models_dict.items():
            rows.append({
                'Set': label,
                'Momentum Range': mr['name'],
                'Model': model_display_names.get(model_type, model_type),
                'Macro AUC': auc_val,
            })
    return rows

rows_no = build_macro_auc_rows(macro_auc_no_dpg, 'NO DPG CUTS')
rows_dpg = build_macro_auc_rows(macro_auc_dpg, 'WITH DPG CUTS')

if rows_no:
    df_no = pd.DataFrame(rows_no)
    print("MACRO AUC SUMMARY TABLE (NO DPG CUTS):\n")
    print(df_no[['Momentum Range', 'Model', 'Macro AUC']].to_string(index=False))
    print()
else:
    df_no = None
    print("No macro AUC data for NO DPG CUTS.\n")

if rows_dpg:
    df_dpg = pd.DataFrame(rows_dpg)
    print("MACRO AUC SUMMARY TABLE (WITH DPG CUTS):\n")
    print(df_dpg[['Momentum Range', 'Model', 'Macro AUC']].to_string(index=False))
    print()
else:
    df_dpg = None
    print("No macro AUC data for WITH DPG CUTS.\n")

# --- Rankings ---

def rank_macro_auc(df, label):
    if df is None or df.empty:
        print(f"No data for MACRO AUC RANKING ({label}).\n")
        return None
    r = df.copy().sort_values('Macro AUC', ascending=False).reset_index(drop=True)
    r.insert(0, 'Rank', np.arange(1, len(r) + 1))
    print(f"MACRO AUC RANKING ({label}):\n")
    print(r[['Rank', 'Model', 'Momentum Range', 'Macro AUC']].to_string(index=False))
    print()
    return r

rank_no  = rank_macro_auc(df_no,  'NO DPG CUTS')
rank_dpg = rank_macro_auc(df_dpg, 'WITH DPG CUTS')

# Combined ranking
if df_no is not None or df_dpg is not None:
    combined_parts = []
    if df_no is not None:
        combined_parts.append(df_no.assign(Set='NO DPG CUTS'))
    if df_dpg is not None:
        combined_parts.append(df_dpg.assign(Set='WITH DPG CUTS'))
    df_all = pd.concat(combined_parts, ignore_index=True)
    rank_all = df_all.copy().sort_values('Macro AUC', ascending=False).reset_index(drop=True)
    rank_all.insert(0, 'Rank', np.arange(1, len(rank_all) + 1))
    print("MACRO AUC RANKING (NO DPG CUTS + WITH DPG CUTS COMBINED):\n")
    print(rank_all[['Rank', 'Set', 'Model', 'Momentum Range', 'Macro AUC']].to_string(index=False))
    print()
else:
    print("No data for combined MACRO AUC ranking.\n")

# ============================================================================
# PART 4: MACRO AUC HEATMAPS (Models × Momentum Ranges)
# ============================================================================

print(f"\n{'='*80}")
print("MACRO AUC HEATMAPS (Models × Momentum Ranges)")
print(f"{'='*80}\n")

def plot_auc_heatmap(macro_dict, title_suffix):
    if not macro_dict:
        print(f"No macro AUC data for heatmap ({title_suffix}).")
        return

    # Data processing remains the same
    heat_rows = []
    for mr_key, models_dict in macro_dict.items():
        mr = momentum_by_key[mr_key]
        for model_type, auc_val in models_dict.items():
            heat_rows.append({
                'Model': model_display_names.get(model_type, model_type),
                'Momentum Range': mr['name'],
                'Macro AUC': auc_val,
            })

    heat_df = pd.DataFrame(heat_rows)
    pivot = heat_df.pivot_table(values='Macro AUC', index='Model', columns='Momentum Range', aggfunc='mean')
    
    # Ensure columns are in the same order every time
    desired_cols = [momentum_by_key[k]['name'] for k in macro_dict.keys()]
    pivot = pivot.reindex(columns=desired_cols)
    
    fig, ax = plt.subplots(figsize=(8, 6)) 

    sns.heatmap(pivot, 
                annot=True, 
                fmt='.3f', 
                cmap='RdYlGn', 
                ax=ax,
                square=True, 
                cbar_kws={'label': 'Macro AUC', 'shrink': 0.8}, 
                vmin=0.7, 
                vmax=1.0, 
                linewidths=0.5)

    ax.set_xlabel('Momentum Range', fontsize=11, fontweight='bold')
    ax.set_ylabel('Model', fontsize=11, fontweight='bold')
    ax.set_title(f"Macro AUC Heatmap ({title_suffix})", fontsize=12, fontweight='bold')

    plt.subplots_adjust(left=0.2, bottom=0.2) 
    plt.show()

print("Heatmap – NO DPG CUTS:")
plot_auc_heatmap(macro_auc_no_dpg, "NO DPG CUTS")

print("\nHeatmap – WITH DPG CUTS:")
plot_auc_heatmap(macro_auc_dpg, "WITH DPG CUTS")

# ============================================================================
# PART 5: MACRO AUC SUMMARY STATISTICS BY MODEL
# ============================================================================

print(f"\n{'='*80}")
print("MACRO AUC SUMMARY STATISTICS BY MODEL")
print(f"{'='*80}\n")

def macro_auc_stats_by_model(macro_dict, label):
    rows = []
    for model_type in MODEL_TYPES:
        model_name = model_display_names.get(model_type, model_type)
        vals = []
        for mr_key, models_dict in macro_dict.items():
            if model_type in models_dict:
                vals.append(models_dict[model_type])
        if vals:
            vals = np.array(vals)
            rows.append({
                'Model': model_name,
                'Avg Macro AUC': float(np.mean(vals)),
                'Min Macro AUC': float(np.min(vals)),
                'Max Macro AUC': float(np.max(vals)),
                'Std Dev': float(np.std(vals)),
            })
    if not rows:
        print(f"No macro AUC data for summary ({label}).\n")
        return
    df = pd.DataFrame(rows).sort_values('Avg Macro AUC', ascending=False)
    print(f"MACRO AUC SUMMARY STATISTICS BY MODEL ({label}):\n")
    print(df.to_string(index=False))
    print()

macro_auc_stats_by_model(macro_auc_no_dpg, "NO DPG CUTS")
macro_auc_stats_by_model(macro_auc_dpg, "WITH DPG CUTS")

print(f"{'='*80}")
print("✓ SECTION 5C COMPLETE: ROC / AUC Analysis with & without DPG cuts")
print(f"{'='*80}\n")


## Section 5D: Efficiency, Purity, F1 & Feature Importance

In [ ]:
# ============================================================================
# SECTION 5D: EFFICIENCY, PURITY, F1 & FEATURE IMPORTANCE (WITH / WITHOUT DPG)
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 5D: EFFICIENCY, PURITY, F1-SCORE & FEATURE IMPORTANCE (WITH / WITHOUT DPG CUTS)")
print(f"{'#'*80}\n")

# Build a convenience map from key -> momentum_range dict
momentum_by_key = {mr['key']: mr for mr in MOMENTUM_RANGES}

# Separate keys: no DPG vs DPG
base_keys = [mr['key'] for mr in MOMENTUM_RANGES if not mr.get('apply_dpg_cuts', False)]
dpg_keys  = [mr['key'] for mr in MOMENTUM_RANGES if mr.get('apply_dpg_cuts', False)]

print("Non-DPG momentum ranges:", base_keys)
print("DPG momentum ranges    :", dpg_keys)

model_display_names = {
    'JAX_SimpleNN':      'SimpleNN',
    'JAX_DNN':           'DNN',
    'JAX_FSE_Attention': 'FSE+Attention',
    'LightGBM':          'LightGBM',
    'XGBoost':           'XGBoost',
}

# ============================================================================
# PART 1: EFFICIENCY, PURITY, F1 PER PARTICLE (PRINTED) – WITH/WITHOUT DPG
# ============================================================================

print(f"\n{'='*80}")
print("EFFICIENCY & PURITY PER PARTICLE TYPE (All models, with F1-score)")
print(f"{'='*80}\n")

efficiency_purity_data = []   # will hold both DPG and non-DPG

for mr_key, mr_container in all_results_by_model_and_range.items():
    mr = momentum_by_key.get(mr_key, None)
    if mr is None:
        continue

    # Tag whether this is DPG or not
    dpg_flag = bool(mr.get('apply_dpg_cuts', False))
    dpg_label = "WITH DPG cuts" if dpg_flag else "NO DPG cuts"

    if 'models' not in mr_container:
        continue

    for model_type in MODEL_TYPES:
        if model_type not in mr_container['models']:
            continue

        results = mr_container['models'][model_type]
        if 'y_test' not in results or 'y_pred_test' not in results:
            continue

        y_test = np.array(results['y_test'])
        y_pred = np.array(results['y_pred_test'])

        print(f"\n{'-'*80}")
        print(f"{mr['name']} ({dpg_label}) - {model_display_names.get(model_type, model_type)}")
        print(f"{'-'*80}\n")

        print(f"{'Particle':<12} {'Efficiency':<15} {'Purity':<15} {'F1-Score':<12} {'Support':<10}")
        print(f"{'-'*64}")

        for i, particle_name in enumerate(PARTICLE_NAMES):
            true_positives = np.sum((y_test == i) & (y_pred == i))
            false_negatives = np.sum((y_test == i) & (y_pred != i))
            false_positives = np.sum((y_test != i) & (y_pred == i))

            eff = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0.0
            pur = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0.0
            f1  = 2 * (eff * pur) / (eff + pur) if (eff + pur) > 0 else 0.0
            support = int(np.sum(y_test == i))

            efficiency_purity_data.append({
                'Momentum Key': mr_key,
                'Momentum Range': mr['name'],
                'Apply DPG Cuts': dpg_flag,
                'Model Type': model_type,
                'Particle': particle_name,
                'Efficiency': eff,
                'Purity': pur,
                'F1-Score': f1,
                'Support': support,
            })

            print(f"{particle_name:<12} {eff:<15.4f} {pur:<15.4f} {f1:<12.4f} {support:<10d}")

efficiency_purity_df = pd.DataFrame(efficiency_purity_data)

# ============================================================================
# PART 2: EFFICIENCY vs PURITY TRADE-OFF (SCATTER) – NO DPG / WITH DPG
# ============================================================================

def plot_eff_pur_tradeoff(eff_df, title_suffix):
    print(f"\n{'='*80}")
    print(f"EFFICIENCY vs PURITY TRADE-OFF {title_suffix}")
    print(f"{'='*80}\n")

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()

    for ax_idx, model_type in enumerate(MODEL_TYPES):
        ax = axes[ax_idx]
        model_data = eff_df[eff_df['Model Type'] == model_type]

        for particle in PARTICLE_NAMES:
            pdata = model_data[model_data['Particle'] == particle]
            effs = pdata['Efficiency'].values
            purs = pdata['Purity'].values

            ax.scatter(effs, purs, s=80, alpha=0.7,
                       label=particle)

        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1)
        ax.set_xlabel('Efficiency (Recall)', fontsize=10, fontweight='bold')
        ax.set_ylabel('Purity (Precision)', fontsize=10, fontweight='bold')
        ax.set_title(model_display_names.get(model_type, model_type),
                     fontsize=11, fontweight='bold')
        ax.set_xlim([0, 1.05])
        ax.set_ylim([0, 1.05])
        ax.grid(alpha=0.3)
        ax.legend(loc='lower left', fontsize=9)

    # hide any unused axes if MODEL_TYPES < 6
    for extra_ax in axes[len(MODEL_TYPES):]:
        extra_ax.set_visible(False)

    plt.suptitle(f'Efficiency vs Purity Trade-off (All Particles, All Ranges, All Models) {title_suffix}',
                 fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

# No DPG cuts
plot_eff_pur_tradeoff(efficiency_purity_df[~efficiency_purity_df['Apply DPG Cuts']], "(NO DPG CUTS)")
# With DPG cuts
plot_eff_pur_tradeoff(efficiency_purity_df[efficiency_purity_df['Apply DPG Cuts']], "(WITH DPG CUTS)")

# ============================================================================
# PART 3: EFFICIENCY / PURITY / F1 – COMPARISON BY MODEL (NO DPG vs DPG)
# ============================================================================

def plot_metric_comparison(metric_col, metric_label):
    for dpg_flag, label in [(False, "NO DPG CUTS"), (True, "WITH DPG CUTS")]:
        subset = efficiency_purity_df[efficiency_purity_df['Apply DPG Cuts'] == dpg_flag]

        print(f"\n{'='*80}")
        print(f"{metric_label.upper()} COMPARISON ACROSS ALL MODELS (Per Particle, {label})")
        print(f"{'='*80}\n")

        fig, axes = plt.subplots(1, len(PARTICLE_NAMES), figsize=(6 * len(PARTICLE_NAMES), 5))

        for p_idx, particle_name in enumerate(PARTICLE_NAMES):
            ax = axes[p_idx]
            pdata = subset[subset['Particle'] == particle_name]

            model_vals = {m: [] for m in MODEL_TYPES}
            for model_type in MODEL_TYPES:
                mdata = pdata[pdata['Model Type'] == model_type]
                if not mdata.empty:
                    model_vals[model_type].append(mdata[metric_col].mean())

            x_pos = np.arange(len(MODEL_TYPES))
            vals = [np.mean(model_vals[m]) if model_vals[m] else 0 for m in MODEL_TYPES]
            colors = [model_colors.get(m, '#999999') for m in MODEL_TYPES]

            bars = ax.bar(x_pos, vals, color=colors, alpha=0.8,
                          edgecolor='black', linewidth=1.5)
            for bar in bars:
                h = bar.get_height()
                if h > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2., h + 0.02,
                            f"{h:.3f}", ha='center', va='bottom',
                            fontsize=9, fontweight='bold')

            ax.set_ylabel(f'{metric_label} (Avg)', fontsize=10, fontweight='bold')
            ax.set_title(particle_name, fontsize=11, fontweight='bold')
            ax.set_xticks(x_pos)
            ax.set_xticklabels([model_display_names.get(m, m) for m in MODEL_TYPES],
                               rotation=45, ha='right', fontsize=9)
            ax.set_ylim([0, 1.05])
            ax.grid(axis='y', alpha=0.3)

        plt.suptitle(f'Average {metric_label} by Model (All Momentum Ranges, All Models, {label})',
                     fontsize=13, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.show()

# Efficiency, purity, F1 comparisons
plot_metric_comparison('Efficiency', 'Efficiency')
plot_metric_comparison('Purity', 'Purity')
plot_metric_comparison('F1-Score', 'F1-Score')

# ============================================================================
# PART 4: COMPREHENSIVE F1 SCORE ANALYSIS (RANKING, HEATMAP, BY RANGE)
# ============================================================================

print(f"\n{'='*80}")
print("COMPREHENSIVE F1-SCORE ANALYSIS (WITH / WITHOUT DPG CUTS)")
print(f"{'='*80}\n")

def build_f1_ranking(subset_df, label):
    print(f"\n{'─'*80}")
    print(f"F1-SCORE RANKING (Model × Momentum Range, Averaged Over Particles) – {label}")
    print(f"{'─'*80}\n")

    rows = []
    for mr_key in subset_df['Momentum Key'].unique():
        mr_name = momentum_by_key[mr_key]['name']
        mr_part = subset_df[subset_df['Momentum Key'] == mr_key]
        for model_type in MODEL_TYPES:
            mdata = mr_part[mr_part['Model Type'] == model_type]
            if mdata.empty:
                continue
            avg_eff = mdata['Efficiency'].mean()
            avg_pur = mdata['Purity'].mean()
            avg_f1  = mdata['F1-Score'].mean()
            rows.append({
                'Model Type': model_type,
                'Model': model_display_names.get(model_type, model_type),
                'Momentum Key': mr_key,
                'Momentum Range': mr_name,
                'Avg Efficiency': avg_eff,
                'Avg Purity': avg_pur,
                'Avg F1-Score': avg_f1,
            })

    if not rows:
        print("No data.")
        return None

    df = pd.DataFrame(rows).sort_values('Avg F1-Score', ascending=False).reset_index(drop=True)
    df['Rank'] = np.arange(1, len(df) + 1)

    print(df[['Rank', 'Model', 'Momentum Range', 'Avg Efficiency', 'Avg Purity', 'Avg F1-Score']].to_string(index=False))
    print()
    return df

# Rankings: no DPG, with DPG, and combined
f1_ranking_no_dpg  = build_f1_ranking(efficiency_purity_df[~efficiency_purity_df['Apply DPG Cuts']], "NO DPG CUTS")
f1_ranking_dpg     = build_f1_ranking(efficiency_purity_df[efficiency_purity_df['Apply DPG Cuts']], "WITH DPG CUTS")
f1_ranking_all     = build_f1_ranking(efficiency_purity_df, "ALL RANGES (WITH & WITHOUT DPG CUTS)")

def plot_f1_heatmap(f1_df, label):
    if f1_df is None or f1_df.empty:
        return
    
    print(f"\n{'─'*80}")
    print(f"F1-SCORE HEATMAP (Models vs Momentum Ranges) – {label}")
    print(f"{'─'*80}\n")

    pivot = f1_df.pivot_table(
        values='Avg F1-Score',
        index='Model',
        columns='Momentum Range',
        aggfunc='mean'
    )

    fig, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(pivot, 
                annot=True, 
                fmt='.3f', 
                cmap='RdYlGn', 
                ax=ax,
                square=True,
                cbar_kws={'label': 'F1-Score', 'shrink': 0.8}, 
                vmin=0, 
                vmax=1, 
                linewidths=0.5)

    ax.set_xlabel('Momentum Range', fontsize=11, fontweight='bold')
    ax.set_ylabel('Model', fontsize=11, fontweight='bold')
    ax.set_title(f'F1-Score Heatmap – {label}', fontsize=12, fontweight='bold')

    plt.subplots_adjust(left=0.2, bottom=0.2, right=0.95, top=0.9)
    plt.show()

# Heatmaps: no DPG & with DPG
plot_f1_heatmap(f1_ranking_no_dpg, "NO DPG CUTS")
plot_f1_heatmap(f1_ranking_dpg, "WITH DPG CUTS")

# F1 comparison per momentum range (bar plots), no DPG / with DPG
def plot_f1_by_range(f1_df, label):
    if f1_df is None or f1_df.empty:
        return

    mr_keys = list(f1_df['Momentum Key'].unique())
    fig, axes = plt.subplots(1, len(mr_keys), figsize=(7 * len(mr_keys), 5))

    if len(mr_keys) == 1:
        axes = [axes]

    for ax, mr_key in zip(axes, mr_keys):
        mr_name = momentum_by_key[mr_key]['name']
        df_mr = f1_df[f1_df['Momentum Key'] == mr_key]

        x_pos = np.arange(len(MODEL_TYPES))
        f1_vals = []
        colors = []
        for model_type in MODEL_TYPES:
            mdata = df_mr[df_mr['Model Type'] == model_type]
            if not mdata.empty:
                f1_vals.append(mdata['Avg F1-Score'].iloc[0])
                colors.append(model_colors.get(model_type, '#999999'))
            else:
                f1_vals.append(0.0)
                colors.append('#CCCCCC')

        bars = ax.bar(x_pos, f1_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        for bar in bars:
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2., h + 0.01,
                        f"{h:.3f}", ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

        ax.set_ylabel('F1-Score', fontsize=10, fontweight='bold')
        ax.set_title(mr_name, fontsize=11, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels([model_display_names.get(m, m) for m in MODEL_TYPES],
                           rotation=45, ha='right', fontsize=9)
        ax.set_ylim([0, 1.0])
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle(f'F1-Score Comparison (All Models, {label})',
                 fontsize=13, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

plot_f1_by_range(f1_ranking_no_dpg,  "All Momentum Ranges – NO DPG CUTS")
plot_f1_by_range(f1_ranking_dpg,     "All Momentum Ranges – WITH DPG CUTS")

# ============================================================================
# PART 5: F1 BY PARTICLE TYPE (RANKED) – NO DPG & WITH DPG
# ============================================================================

def print_f1_by_particle(subset_df, label):
    print(f"\n{'─'*80}")
    print(f"F1-SCORE BY PARTICLE TYPE (All Models, All Ranges) – {label}")
    print(f"{'─'*80}\n")

    rows = []
    for particle_name in PARTICLE_NAMES:
        psub = subset_df[subset_df['Particle'] == particle_name]
        for model_type in MODEL_TYPES:
            mdata = psub[psub['Model Type'] == model_type]
            if mdata.empty:
                continue
            avg_f1 = mdata['F1-Score'].mean()
            rows.append({
                'Particle': particle_name,
                'Model Type': model_type,
                'Model': model_display_names.get(model_type, model_type),
                'Avg F1-Score': avg_f1,
            })

    if not rows:
        print("No data.\n")
        return

    df = pd.DataFrame(rows).sort_values('Avg F1-Score', ascending=False).reset_index(drop=True)
    df['Rank'] = np.arange(1, len(df) + 1)
    print(df[['Rank', 'Particle', 'Model', 'Avg F1-Score']].to_string(index=False))
    print()

print_f1_by_particle(efficiency_purity_df[~efficiency_purity_df['Apply DPG Cuts']], "NO DPG CUTS")
print_f1_by_particle(efficiency_purity_df[efficiency_purity_df['Apply DPG Cuts']],    "WITH DPG CUTS")

# ============================================================================
# PART 6: FEATURE IMPORTANCE VISUALISATION (TREE MODELS, WITH/WITHOUT DPG)
# ============================================================================

print(f"\n{'='*80}")
print("FEATURE IMPORTANCE VISUALISATION (Top 15 Features, Tree-Based Models)")
print(f"{'='*80}\n")

def collect_feature_importances(dpg_flag, label):
    rows = []
    for mr_key, mr_container in all_results_by_model_and_range.items():
        mr = momentum_by_key.get(mr_key, None)
        if mr is None or bool(mr.get('apply_dpg_cuts', False)) != dpg_flag:
            continue
        if 'models' not in mr_container:
            continue

        for model_type in ['LightGBM', 'XGBoost']:
            if model_type not in mr_container['models']:
                continue
            results = mr_container['models'][model_type]
            model_obj = results.get('model', None)

            importances = None
            if hasattr(model_obj, 'feature_importances_'):
                importances = getattr(model_obj, 'feature_importances_')

            if importances is None:
                continue

            importances = np.array(importances)
            # Align size
            if len(importances) < len(TRAINING_FEATURES):
                importances = np.pad(importances, (0, len(TRAINING_FEATURES) - len(importances)), mode='constant')
            elif len(importances) > len(TRAINING_FEATURES):
                importances = importances[:len(TRAINING_FEATURES)]

            if np.sum(importances) > 0:
                importances = importances / np.sum(importances) * 100.0

            rows.append({
                'Momentum Key': mr_key,
                'Momentum Range': mr['name'],
                'Apply DPG Cuts': dpg_flag,
                'Model Type': model_type,
                'Model': model_display_names.get(model_type, model_type),
                'Importances': importances,
            })

    if not rows:
        print(f"No feature importance data found for {label}.")
        return

    # Plot: up to 6 subplots (2 x 3)
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    plot_idx = 0

    for row in rows:
        if plot_idx >= 6:
            break
        ax = axes[plot_idx]
        importances = row['Importances']
        mr_name = row['Momentum Range']
        model_name = row['Model']

        sorted_idx = np.argsort(importances)[-15:]
        sorted_importances = importances[sorted_idx]
        sorted_names = [TRAINING_FEATURES[i] for i in sorted_idx]

        ax.barh(range(len(sorted_importances)), sorted_importances,
                color=model_colors.get(row['Model Type'], '#999999'),
                alpha=0.8, edgecolor='black', linewidth=1.5)
        ax.set_yticks(range(len(sorted_importances)))
        ax.set_yticklabels(sorted_names, fontsize=9)
        ax.set_xlabel('Importance (%)', fontsize=10, fontweight='bold')
        ax.set_title(f"{model_name}\n{mr_name} ({label})",
                     fontsize=11, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

        plot_idx += 1

    for idx in range(plot_idx, 6):
        axes[idx].set_visible(False)

    plt.suptitle(f'Feature Importance (Top 15 Features, Tree-Based Models) – {label}',
                 fontsize=13, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.show()

collect_feature_importances(False, "NO DPG CUTS")
collect_feature_importances(True,  "WITH DPG CUTS")

# ============================================================================
# PART 7: OVERALL SUMMARY STATISTICS (ALL MODELS, WITH/WITHOUT DPG)
# ============================================================================

print(f"\n{'='*80}")
print("OVERALL SUMMARY STATISTICS (All Models, All Ranges – WITH / WITHOUT DPG CUTS)")
print(f"{'='*80}\n")

def print_summary(subset_df, label):
    print(f"\n{'─'*80}")
    print(f"SUMMARY STATISTICS – {label}")
    print(f"{'─'*80}\n")

    rows = []
    for model_type in MODEL_TYPES:
        mdata = subset_df[subset_df['Model Type'] == model_type]
        if mdata.empty:
            continue
        rows.append({
            'Model': model_display_names.get(model_type, model_type),
            'Avg Efficiency': mdata['Efficiency'].mean(),
            'Avg Purity': mdata['Purity'].mean(),
            'Avg F1-Score': mdata['F1-Score'].mean(),
            'Min F1-Score': mdata['F1-Score'].min(),
            'Max F1-Score': mdata['F1-Score'].max(),
        })

    if not rows:
        print("No data.\n")
        return

    df = pd.DataFrame(rows).sort_values('Avg F1-Score', ascending=False)
    print(df.to_string(index=False))
    print()

print_summary(efficiency_purity_df[~efficiency_purity_df['Apply DPG Cuts']], "NO DPG CUTS")
print_summary(efficiency_purity_df[efficiency_purity_df['Apply DPG Cuts']],    "WITH DPG CUTS")

print(f"\n{'='*80}")
print("✓ SECTION 5D COMPLETE: Efficiency, Purity, F1-Score & Feature Importance (With / Without DPG Cuts)")
print(f"{'='*80}\n")


## Section 5E: Detector-mode Performance Comparison

In [ ]:
# ============================================================================
# SECTION 5E: DETECTOR MODES PERFORMANCE
# ============================================================================

print("\n" + "#"*90)
print("SECTION 5E: DETECTOR MODE PERFORMANCE ANALYSIS")
print("#"*90)

detector_metrics = []

MODE_MAP = {0:'NONE',1:'TPC_ONLY',2:'TOF_ONLY',3:'TPC_TOF'}

# Map momentum keys → readable names
momentum_name_map = {r['key']: r['name'] for r in MOMENTUM_RANGES}

print("\nScanning detector performance across all models and momentum ranges...\n")

for mr_key, container in all_results_by_model_and_range.items():

    if 'models' not in container:
        continue

    print("─"*90)
    print(f"Momentum Range : {momentum_name_map.get(mr_key, mr_key)}")
    print("─"*90)

    if 'detector_modes_test' not in container['preprocessing']:
        print("⚠ detector_modes_test not available\n")
        continue

    det_modes_full = np.array(container['preprocessing']['detector_modes_test'])

    # Show detector composition
    print("Detector composition:")
    for code,mode_name in MODE_MAP.items():
        count = np.sum(det_modes_full == code)
        pct = 100*count/len(det_modes_full)
        print(f"  {mode_name:10s} : {count:10,d}  ({pct:5.2f}%)")

    print()

    # Evaluate models
    for model_name in MODEL_TYPES:

        if model_name not in container['models']:
            continue

        results = container['models'][model_name]

        if 'y_pred_test' not in results:
            continue

        y_test = np.array(results['y_test'])
        y_pred = np.array(results['y_pred_test'])

        safe_len = min(len(y_test), len(y_pred), len(det_modes_full))

        det_modes = det_modes_full[:safe_len]
        y_test_slice = y_test[:safe_len]
        y_pred_slice = y_pred[:safe_len]

        print(f"{model_name:20s} | Samples analysed: {safe_len:,}")

        for code,mode_name in MODE_MAP.items():

            mask = det_modes == code
            count = np.sum(mask)

            if count < 100:
                continue

            acc = accuracy_score(y_test_slice[mask], y_pred_slice[mask])

            print(f"   {mode_name:10s} → Accuracy: {acc:.4f}   (N={count:,})")

            detector_metrics.append({
                'range_key': mr_key,
                'range': momentum_name_map.get(mr_key, mr_key),
                'model': model_name,
                'mode': mode_name,
                'accuracy': acc,
                'n_samples': count
            })

    print()


# Convert to DataFrame
df_metrics = pd.DataFrame(detector_metrics)

print("\nCollected detector metrics:")
print(df_metrics.head())

# ============================================================================
# BUILD MOMENTUM GROUPS FROM SECTION 0
# ============================================================================

no_cut_ranges = [r['key'] for r in MOMENTUM_RANGES if not r['apply_dpg_cuts']]
dpg_ranges = [r['key'] for r in MOMENTUM_RANGES if r['apply_dpg_cuts']]

range_groups = {
    "No-Cut Ranges": no_cut_ranges,
    "DPG-Cut Ranges": dpg_ranges
}

# ============================================================================
# TPC_ONLY PERFORMANCE TABLE
# ============================================================================

print("\n" + "="*100)
print("TPC_ONLY ACCURACY BY MOMENTUM RANGE")
print("="*100)

tpc_data = df_metrics[df_metrics['mode']=='TPC_ONLY']

for group_name, range_keys in range_groups.items():

    print(f"\n{group_name:^100}")
    print("-"*100)

    header = f"{'Model':18s}"
    for r in range_keys:
        header += f"{r:>12s}"
    print(header)
    print("-"*100)

    for model in MODEL_TYPES:

        row = f"{model.replace('JAX_',''):18s}"

        for r in range_keys:

            match = tpc_data[
                (tpc_data['model']==model) &
                (tpc_data['range_key']==r)
            ]

            if len(match) > 0:
                row += f"{match.iloc[0]['accuracy']:12.4f}"
            else:
                row += f"{'--':>12}"

        print(row)

# ============================================================================
# DETECTOR MODE COMPARISON PLOTS WITH/WITHOUT DPG CUTS
# ============================================================================

print("\nGenerating detector performance plots...\n")

# Split ranges using Section 0 configuration
no_cut_ranges = [r['key'] for r in MOMENTUM_RANGES if not r['apply_dpg_cuts']]
dpg_ranges = [r['key'] for r in MOMENTUM_RANGES if r['apply_dpg_cuts']]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

modes = ['TPC_ONLY', 'NONE', 'TPC_TOF']
titles = ['TPC Only', 'No Detector', 'TPC + TOF']

groups = [
    ("No DPG Cuts", no_cut_ranges),
    ("DPG Cuts Applied", dpg_ranges)
]

for row_idx, (group_name, range_keys) in enumerate(groups):

    group_data = df_metrics[df_metrics['range_key'].isin(range_keys)]

    for col_idx, (mode, title) in enumerate(zip(modes, titles)):

        ax = axes[row_idx, col_idx]

        mode_data = group_data[group_data['mode'] == mode]

        model_avgs = []

        for model in MODEL_TYPES:
            m = mode_data[mode_data['model'] == model]

            if len(m) > 0:
                model_avgs.append(m['accuracy'].mean())
            else:
                model_avgs.append(np.nan)

        x = np.arange(len(MODEL_TYPES))
        colors = [model_colors[m] for m in MODEL_TYPES]

        ax.bar(
            x,
            model_avgs,
            color=colors,
            edgecolor='black',
            linewidth=1.2,
            alpha=0.85
        )

        labels = [m.replace('JAX_', '') for m in MODEL_TYPES]

        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=45, ha='right')

        ax.set_title(f"{title}\n{group_name}", fontsize=13, fontweight='bold')
        ax.set_ylabel("Accuracy")

        ax.set_ylim(0.7, 1.02)
        ax.grid(True, axis='y', alpha=0.3)

        # Annotate bars
        for i, val in enumerate(model_avgs):
            if not np.isnan(val):
                ax.text(i, val + 0.003, f"{val:.3f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# ============================================================================
# MODEL PERFORMANCE COMPARISON
# ============================================================================

print("\n" + "="*90)
print("MODEL PERFORMANCE COMPARISON")
print("="*90)

# Split ranges using Section 1 config
no_cut_ranges = [r['key'] for r in MOMENTUM_RANGES if not r['apply_dpg_cuts']]
dpg_ranges = [r['key'] for r in MOMENTUM_RANGES if r['apply_dpg_cuts']]

groups = [
    ("No DPG Cuts", no_cut_ranges),
    ("DPG Cuts Applied", dpg_ranges)
]

comparison_pairs = [
    ("JAX_SimpleNN", "JAX_DNN"),
    ("JAX_DNN", "JAX_FSE_Attention"),
    ("JAX_FSE_Attention", "LightGBM"),
    ("LightGBM", "XGBoost")
]

for group_name, range_keys in groups:

    print(f"\n{group_name}")
    print("-"*90)

    group_data = df_metrics[df_metrics['range_key'].isin(range_keys)]

    model_means = {}

    for model in MODEL_TYPES:
        m = group_data[group_data['model']==model]

        if len(m) > 0:
            model_means[model] = m['accuracy'].mean()
        else:
            model_means[model] = np.nan

    # Print absolute accuracies
    print("\nAverage Accuracy by Model:")
    for model in MODEL_TYPES:
        label = model.replace('JAX_','')
        acc = model_means[model]

        if not np.isnan(acc):
            print(f"  {label:15s}: {acc:.4f}")

    print("\nModel Improvements:")

    for base, improved in comparison_pairs:

        base_acc = model_means.get(base, np.nan)
        new_acc = model_means.get(improved, np.nan)

        if np.isnan(base_acc) or np.isnan(new_acc):
            continue

        diff = new_acc - base_acc
        pct = (diff / base_acc) * 100

        base_label = base.replace('JAX_','')
        new_label = improved.replace('JAX_','')

        print(f"  {new_label:15s} vs {base_label:15s}: "
              f"{diff:+.4f} accuracy ({pct:+.2f}%)")

print("\n✓ Model comparison summary complete")

print("\n✓ Detector mode plots generated (No DPG vs DPG comparison)")



## Section 5F: Bayesian Comparison & Analysis

In [ ]:
# ============================================================================
# SECTION 5F: BAYESIAN PID AVAILABILITY & ML MODEL COMPARISON
# ============================================================================

print(f"\n{'#'*80}")
print("SECTION 5F: BAYESIAN PID AVAILABILITY & MODEL COMPARISON")
print(f"{'#'*80}\n")

bayes_features = DETECTOR_GROUPS['bayes']

# ============================================================================
# PART 1: BAYESIAN AVAILABILITY (GLOBAL)
# ============================================================================

print("\n" + "="*80)
print("BAYESIAN PID AVAILABILITY (FULL DATASET)")
print("="*80)

total_rows = len(df)

for feat in bayes_features:
    available = (df[feat] != 0).sum()
    pct = available / total_rows * 100
    print(f"{feat:<20} {available:>10,} ({pct:6.2f}%)")

bayes_complete_mask = (df[bayes_features] != 0).all(axis=1)

complete_count = bayes_complete_mask.sum()
complete_pct = complete_count / total_rows * 100

print("\nComplete Bayesian availability:")
print(f"Tracks with all 4 Bayesian values: {complete_count:,} ({complete_pct:.2f}%)")

# ============================================================================
# PART 2: AVAILABILITY PER MOMENTUM RANGE
# ============================================================================

print("\n" + "="*80)
print("BAYESIAN AVAILABILITY PER MOMENTUM RANGE")
print("="*80)

for mr in MOMENTUM_RANGES:
    df_range = df[(df['p'] >= mr['min']) & (df['p'] < mr['max'])]
    if mr['apply_dpg_cuts']:
        df_range = apply_dpg_track_cuts(df_range)
    mask = (df_range[bayes_features] != 0).all(axis=1)

    print(f"\n{mr['name']}")
    print(f"Tracks: {len(df_range):,}")
    print(f"Complete Bayesian: {mask.sum():,} ({mask.mean()*100:.2f}%)")

# ============================================================================
# PART 3: MODEL vs BAYESIAN COMPARISON
# ============================================================================

print("\n" + "="*80)
print("MODEL vs BAYESIAN PERFORMANCE")
print("="*80)

comparison_data = []

for mr in MOMENTUM_RANGES:

    key = mr["key"]
    if key not in all_results_by_model_and_range:
        continue

    data = all_results_by_model_and_range[key]
    preprocessing = data.get("preprocessing", {})
    models = data.get("models", {})

    if len(models) == 0:
        continue

    reference_model = MODEL_TYPES[0]
    if reference_model not in models:
        continue

    y_test = np.array(models[reference_model]["y_test"])
    test_len = len(y_test)

    bayes_probs = preprocessing.get("bayes_pred_original_test")
    if bayes_probs is None:
        continue

    bayes_probs = np.array(bayes_probs)
    if len(bayes_probs) != test_len:
        bayes_probs = bayes_probs[:test_len]

    bayes_pred = np.argmax(bayes_probs, axis=1)

    bayes_mask = preprocessing.get("bayes_availability_test")
    if bayes_mask is None:
        continue

    bayes_mask = np.array(bayes_mask).astype(bool)
    if len(bayes_mask) != test_len:
        bayes_mask = bayes_mask[:test_len]

    print("\n" + "-"*80)
    print(mr["name"])
    print("-"*80)

    n_real = bayes_mask.sum()
    print(f"Tracks with real Bayesian: {n_real:,}")

    acc_bayes_all = accuracy_score(y_test, bayes_pred)
    print(f"Bayesian Accuracy (all tracks): {acc_bayes_all:.4f}")

    model_accs = {}

    for model in MODEL_TYPES:
        if model not in models:
            continue

        y_pred = np.array(models[model]["y_pred_test"])
        if len(y_pred) != test_len:
            y_pred = y_pred[:test_len]

        acc = accuracy_score(y_test, y_pred)
        improvement = acc - acc_bayes_all

        print(f"{model:<20} {acc:.4f}  Δ {improvement:+.4f}")
        model_accs[model] = acc

    acc_bayes_real = None
    model_real_acc = {}

    if n_real > 0:
        y_test_real = y_test[bayes_mask]
        bayes_real = bayes_pred[bayes_mask]
        acc_bayes_real = accuracy_score(y_test_real, bayes_real)

        print(f"\nBayesian Accuracy (real only): {acc_bayes_real:.4f}")

        for model in MODEL_TYPES:
            if model not in models:
                continue

            y_pred = np.array(models[model]["y_pred_test"])
            if len(y_pred) != test_len:
                y_pred = y_pred[:test_len]

            y_pred = y_pred[bayes_mask]
            acc = accuracy_score(y_test_real, y_pred)

            print(f"{model:<20} {acc:.4f}")
            model_real_acc[model] = acc

    comparison_data.append({
        "range": mr["name"],
        "apply_dpg": mr["apply_dpg_cuts"],
        "bayes_all": acc_bayes_all,
        "bayes_real": acc_bayes_real if acc_bayes_real else 0,
        "models_all": model_accs,
        "models_real": model_real_acc
    })

# ============================================================================
# PART 4: VISUAL COMPARISON
# ============================================================================

print("\nGenerating comparison plots...")

non_dpg = [d for d in comparison_data if not d["apply_dpg"]]
dpg = [d for d in comparison_data if d["apply_dpg"]]

def plot_accuracy_group(data, title, use_real=False):

    fig, axes = plt.subplots(1, len(data), figsize=(24,7))
    if len(data) == 1:
        axes = [axes]

    for i, d in enumerate(data):

        ax = axes[i]

        if use_real:
            models = list(d["models_real"].keys())
            values = [d["models_real"][m] for m in models]
            bayes_val = d["bayes_real"]
            subtitle = "Real Bayesian Only"
        else:
            models = list(d["models_all"].keys())
            values = [d["models_all"][m] for m in models]
            bayes_val = d["bayes_all"]
            subtitle = "All Tracks"

        labels = models + ["Bayesian"]
        values = values + [bayes_val]
        colors = [model_colors.get(m, "#999999") for m in models] + [model_colors["Bayesian_PID"]]

        x_pos = np.arange(len(labels))

        bars = ax.bar(
            x_pos,
            values,
            color=colors,
            alpha=0.85,
            edgecolor='black',
            linewidth=1.5
        )

        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f"{h:.3f}",
                ha='center',
                va='bottom',
                fontsize=9,
                fontweight='bold'
            )

        ax.set_ylim(0,1.05)
        ax.set_title(f"{d['range']}\n({subtitle})", fontsize=11, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
        ax.grid(axis="y", alpha=0.3)

    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_accuracy_group(non_dpg, "Models vs Bayesian (No DPG Cuts)", use_real=False)
plot_accuracy_group(non_dpg, "Models vs Bayesian (Real Bayesian Only, No DPG Cuts)", use_real=True)
plot_accuracy_group(dpg, "Models vs Bayesian (With DPG Cuts)", use_real=False)
plot_accuracy_group(dpg, "Models vs Bayesian (Real Bayesian Only, With DPG Cuts)", use_real=True)

# ============================================================================
# PART 5: IMPROVEMENT OVER BAYESIAN
# ============================================================================

print("\nGenerating improvement plots...")

def plot_improvement(data, title):

    fig, ax = plt.subplots(figsize=(14,7))

    for model in MODEL_TYPES:

        improvements = []

        for d in data:
            if model in d["models_all"]:
                imp = (d["models_all"][model] - d["bayes_all"]) / d["bayes_all"] * 100
            else:
                imp = np.nan
            improvements.append(imp)

        ax.plot(
            [d["range"] for d in data],
            improvements,
            marker="o",
            label=model,
            color=model_colors.get(model,"#999")
        )

    ax.axhline(0, color="black", linewidth=1.5)
    ax.set_ylabel("Improvement over Bayesian (%)", fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_improvement(non_dpg, "Improvement over Bayesian (No DPG Cuts)")
plot_improvement(dpg, "Improvement over Bayesian (With DPG Cuts)")

print("\n✓ SECTION 5F COMPLETE")
